In [ ]:
# -*- coding: utf-8 -*-
"""
Predictive-profile experiment pipeline for Google Colab.

One command runs the entire workflow:
  1) mount Google Drive (when running in Colab)
  2) import existing HMM bundles when possible
  3) generate a resumable HMM candidate bank
  4) compute data-side predictive metrics
  5) select matched HMM pairs
  6) generate/cache sequence datasets
  7) train GRU and causal Transformer models with per-epoch checkpoints
  8) resume incomplete conditions automatically after a disconnect
  9) run sample-size and empirical-rank analyses
 10) aggregate statistics and create figures/tables

The pipeline never overwrites the user's earlier project. All outputs are written to
predictive_profile_experiment_v1 under Google Drive by default.
"""

from __future__ import annotations

import gc
import hashlib
import json
import logging
import math
import os
import pickle
import random
import re
import shutil
import sys
import time
import traceback
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

try:
    import matplotlib.pyplot as plt
except Exception as exc:  # pragma: no cover
    raise RuntimeError("matplotlib is required") from exc

try:
    from scipy.linalg import eig
except Exception as exc:  # pragma: no cover
    raise RuntimeError("scipy is required") from exc

try:
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except Exception as exc:  # pragma: no cover
    raise RuntimeError("scikit-learn is required") from exc

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.nn.utils.rnn import pack_padded_sequence
except Exception as exc:  # pragma: no cover
    raise RuntimeError("PyTorch is required") from exc


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

@dataclass
class Config:
    # Paths
    DRIVE_ROOT: str = "/content/drive/MyDrive"
    PROJECT_ROOT: str = (
        "/content/drive/MyDrive/predictive_complexity_project/"
        "predictive_profile_experiment_v2"
    )
    EXISTING_HMM_DIR: str = (
        "/content/drive/MyDrive/predictive_complexity_project/"
        "hmm_sameK_transformer_gru_process_comparable_v1/data/regenerated_bundles"
    )
    EXISTING_RESULTS_DIRS: Tuple[str, ...] = (
        "/content/drive/MyDrive/predictive_complexity_project/",
        "/content/drive/MyDrive/predictive_complexity_project/tmlr_additional_experiments/",
    )

    # Reproducibility
    MASTER_SEED: int = 20260715
    DEVICE: str = "auto"  # auto/cpu/cuda
    DETERMINISTIC_TORCH: bool = True

    # HMM family and bank
    K_VALUES: Tuple[int, ...] = (3, 4, 5, 6)
    VOCAB_SIZE: int = 8
    BANK_PER_K: int = 1000
    BANK_SAVE_EVERY: int = 25
    TRANSITION_ALPHA_GRID: Tuple[float, ...] = (0.2, 1.0, 5.0)
    SELF_BIAS_GRID: Tuple[float, ...] = (0.0, 2.0, 8.0)
    EMISSION_ALPHA_GRID: Tuple[float, ...] = (0.1, 0.5, 2.0)
    MIN_STATIONARY_PROB: float = 0.01
    MIN_SYMBOL_PROB: float = 1e-5
    MIN_CONTEXT_GAIN: float = 1e-5
    MAX_HIDDEN_TAU: float = 1e4

    # Predictive metrics
    MAX_CONTEXT: int = 4
    BANK_FUTURE_LENGTHS: Tuple[int, ...] = (1, 2)
    SELECTED_FUTURE_LENGTHS: Tuple[int, ...] = (1, 2, 3)
    SELECTION_FUTURE_LENGTH: int = 2
    ENERGY_LEVELS: Tuple[float, ...] = (0.90, 0.95, 0.99)
    SPECTRAL_TOL: float = 1e-10
    OBS_MIX_MAX_LAG: int = 50

    # Matched-pair selection
    RANK_PAIRS_PER_K: int = 2
    SIGMA_PAIRS_PER_K: int = 1
    CROSS_K_PAIRS: int = 6
    MAX_SELECTED_HMMS: int = 24
    MIN_RANK_CONTRAST: int = 1
    MIN_LOG_SIGMA_CONTRAST: float = 0.35
    MAX_CONTROL_DISTANCE: float = 2.5

    # Raw-ratio safeguards for matched-pair quality (v4 protocol)
    RANK_MAX_GAIN_RATIO: float = 1.25
    RANK_MAX_SIGMA_RATIO: float = 1.25
    RANK_MAX_TAU_RATIO: float = 1.25
    SIGMA_MIN_RATIO: float = 10.0
    SIGMA_MAX_GAIN_RATIO: float = 1.25
    SIGMA_MAX_TAU_RATIO: float = 1.35
    MATCH_PROTOCOL_VERSION: str = "v4_strict_ratio_crossK_reuse"

    # Data generation
    SEQUENCE_LENGTH: int = 300
    N_TRAIN_MAX: int = 1024
    N_TRAIN_MAIN: int = 256
    N_VAL: int = 32
    N_TEST: int = 64
    EVAL_WINDOWS_PER_SEQUENCE: int = 8

    # Neural models
    ARCHITECTURES: Tuple[str, ...] = ("GRU", "Transformer")
    DIMENSIONS: Tuple[int, ...] = (2, 4, 8, 16, 32, 64, 128, 256)
    SEEDS: Tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7)
    BATCH_SIZE: int = 16
    MIN_BATCH_SIZE: int = 2
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 0.0
    MAX_EPOCHS: int = 48
    EARLY_STOP_PATIENCE: int = 8
    MIN_EPOCHS: int = 12
    GRAD_CLIP_NORM: float = 1.0
    TRANSFORMER_HEADS: int = 1
    TRANSFORMER_LAYERS: int = 1
    TRANSFORMER_FF_MULTIPLIER: int = 4
    DROPOUT: float = 0.0
    CHECKPOINT_EVERY_EPOCH: bool = True

    # Optional analyses included in the same run
    RUN_SAMPLE_SIZE_EXPERIMENT: bool = True
    SAMPLE_SIZE_VALUES: Tuple[int, ...] = (64, 128, 256, 512, 1024)
    SAMPLE_SIZE_DIMENSION: int = 128
    SAMPLE_SIZE_SEEDS: Tuple[int, ...] = (1, 2, 3, 4, 5)
    SAMPLE_SIZE_MAX_HMMS: int = 8

    RUN_EMPIRICAL_RANK: bool = True
    EMPIRICAL_RANK_H: int = 4
    EMPIRICAL_RANK_M: int = 1
    EMPIRICAL_NULL_REPEATS: int = 100
    EMPIRICAL_MAX_WINDOWS: int = 100000

    # Pipeline switches
    IMPORT_EXISTING_HMMS: bool = True
    GENERATE_BANK: bool = True
    RUN_TRAINING: bool = True
    RUN_ANALYSIS: bool = True
    CONTINUE_AFTER_ERROR: bool = True
    RETRY_FAILED_CONDITIONS: bool = True
    FORCE_RECOMPUTE_METRICS: bool = False
    FORCE_RESELECT: bool = False
    FORCE_RETRAIN: bool = False

    # Diagnostics
    QUICK_SMOKE_TEST: bool = False

    def apply_smoke_test(self) -> None:
        if not self.QUICK_SMOKE_TEST:
            return
        self.K_VALUES = (3, 4)
        self.BANK_PER_K = 8
        self.RANK_PAIRS_PER_K = 1
        self.SIGMA_PAIRS_PER_K = 0
        self.CROSS_K_PAIRS = 1
        self.MAX_SELECTED_HMMS = 4
        self.N_TRAIN_MAX = 32
        self.N_TRAIN_MAIN = 16
        self.N_VAL = 8
        self.N_TEST = 8
        self.SEQUENCE_LENGTH = 40
        self.EVAL_WINDOWS_PER_SEQUENCE = 2
        self.DIMENSIONS = (4,)
        self.SEEDS = (1,)
        self.MAX_EPOCHS = 2
        self.MIN_EPOCHS = 1
        self.EARLY_STOP_PATIENCE = 1
        self.BATCH_SIZE = 8
        self.RUN_SAMPLE_SIZE_EXPERIMENT = False
        self.EMPIRICAL_NULL_REPEATS = 3
        self.EMPIRICAL_MAX_WINDOWS = 500


# Edit only this dictionary for ordinary Colab use. Unlisted settings keep their defaults.
# Set QUICK_SMOKE_TEST=True for a tiny end-to-end check; return it to False for the full run.
USER_OVERRIDES: Dict[str, Any] = {
    "QUICK_SMOKE_TEST": False,
    # "PROJECT_ROOT": "/content/drive/MyDrive/predictive_complexity_project/predictive_profile_experiment_v1",
    # "BANK_PER_K": 1000,
    # "SEEDS": (1, 2, 3, 4, 5, 6, 7),
}

CFG = Config()
for _name, _value in USER_OVERRIDES.items():
    if not hasattr(CFG, _name):
        raise KeyError(f"Unknown Config override: {_name}")
    setattr(CFG, _name, _value)
if os.environ.get("PP_SMOKE_TEST", "0") == "1":
    CFG.QUICK_SMOKE_TEST = True
CFG.apply_smoke_test()


# -----------------------------------------------------------------------------
# Paths, logging, atomic file operations
# -----------------------------------------------------------------------------

def in_colab() -> bool:
    return "google.colab" in sys.modules


def mount_drive_if_needed(cfg: Config) -> None:
    if not in_colab():
        return
    from google.colab import drive  # type: ignore

    mount_point = "/content/drive"
    if not Path(mount_point, "MyDrive").exists():
        drive.mount(mount_point)


def configure_local_paths_for_non_colab(cfg: Config) -> None:
    """Make smoke tests runnable outside Colab without editing the notebook."""
    if in_colab():
        return
    if cfg.PROJECT_ROOT.startswith("/content/drive/"):
        cfg.PROJECT_ROOT = str(Path.cwd() / "predictive_profile_experiment_v1_local")
    if cfg.EXISTING_HMM_DIR.startswith("/content/drive/"):
        cfg.EXISTING_HMM_DIR = str(Path.cwd() / "nonexistent_existing_hmms")
    cfg.EXISTING_RESULTS_DIRS = tuple()


def project_paths(cfg: Config) -> Dict[str, Path]:
    root = Path(cfg.PROJECT_ROOT)
    paths = {
        "root": root,
        "logs": root / "logs",
        "config": root / "config",
        "bank": root / "candidate_bank",
        "imported": root / "imported_hmms",
        "selected": root / "selected_hmms",
        "datasets": root / "datasets",
        "runs": root / "runs",
        "sample_runs": root / "sample_size_runs",
        "analysis": root / "analysis",
        "figures": root / "analysis" / "figures",
        "tables": root / "analysis" / "tables",
        "failures": root / "failures",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def setup_logger(log_dir: Path) -> logging.Logger:
    logger = logging.getLogger("predictive_profile_pipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )
    stream = logging.StreamHandler(sys.stdout)
    stream.setFormatter(formatter)
    logger.addHandler(stream)
    file_handler = logging.FileHandler(log_dir / "pipeline.log", encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    return logger


def json_default(obj: Any) -> Any:
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    raise TypeError(f"Object of type {type(obj)!r} is not JSON serializable")


def atomic_write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, default=json_default)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def atomic_write_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_save_npz(path: Path, **arrays: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def atomic_torch_save(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def safe_torch_load(path: Path, map_location: str | torch.device = "cpu") -> Any:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def stable_hash(text: str, modulo: int = 2_000_000_000) -> int:
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % modulo


def set_global_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass


def resolve_device(cfg: Config) -> torch.device:
    if cfg.DEVICE == "cpu":
        return torch.device("cpu")
    if cfg.DEVICE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CFG.DEVICE='cuda' but CUDA is not available")
        return torch.device("cuda")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def record_failure(paths: Dict[str, Path], stage: str, identifier: str, exc: BaseException) -> None:
    payload = {
        "stage": stage,
        "identifier": identifier,
        "exception_type": type(exc).__name__,
        "message": str(exc),
        "traceback": traceback.format_exc(),
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", identifier)
    atomic_write_json(paths["failures"] / f"{stage}__{safe_id}.json", payload)


# -----------------------------------------------------------------------------
# HMM utilities
# -----------------------------------------------------------------------------

def row_normalize(matrix: np.ndarray, eps: float = 1e-15) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float64)
    sums = matrix.sum(axis=1, keepdims=True)
    if np.any(sums <= eps):
        raise ValueError("A probability row has zero mass")
    return matrix / sums


def stationary_distribution(T: np.ndarray) -> np.ndarray:
    T = np.asarray(T, dtype=np.float64)
    values, vectors = eig(T.T)
    idx = int(np.argmin(np.abs(values - 1.0)))
    vec = np.real(vectors[:, idx])
    vec = np.abs(vec)
    if vec.sum() <= 0:
        raise ValueError("Could not compute a stationary distribution")
    pi = vec / vec.sum()
    return pi


def canonicalize_hmm(
    T: np.ndarray,
    O: np.ndarray,
    pi: Optional[np.ndarray] = None,
    expected_vocab: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    T = np.asarray(T, dtype=np.float64)
    O = np.asarray(O, dtype=np.float64)
    if T.ndim != 2 or T.shape[0] != T.shape[1]:
        raise ValueError(f"Transition matrix must be square; received {T.shape}")
    K = T.shape[0]
    if O.ndim != 2:
        raise ValueError(f"Emission matrix must be 2D; received {O.shape}")
    if O.shape[0] != K and O.shape[1] == K:
        O = O.T
    if O.shape[0] != K:
        raise ValueError(f"Emission matrix shape {O.shape} is incompatible with K={K}")
    if expected_vocab is not None and O.shape[1] != expected_vocab:
        raise ValueError(
            f"Expected vocabulary size {expected_vocab}, but emission matrix has {O.shape[1]}"
        )
    if np.any(T < -1e-12) or np.any(O < -1e-12):
        raise ValueError("Negative probabilities detected")
    T = row_normalize(np.clip(T, 0.0, None))
    O = row_normalize(np.clip(O, 0.0, None))
    if pi is None:
        pi = stationary_distribution(T)
    else:
        pi = np.asarray(pi, dtype=np.float64).reshape(-1)
        if pi.shape[0] != K:
            raise ValueError("Initial distribution length does not match K")
        pi = np.clip(pi, 0.0, None)
        if pi.sum() <= 0:
            raise ValueError("Initial distribution has zero mass")
        pi = pi / pi.sum()
    return T, O, pi


def validate_hmm(T: np.ndarray, O: np.ndarray, pi: np.ndarray, cfg: Config) -> Tuple[bool, str]:
    try:
        T, O, pi = canonicalize_hmm(T, O, pi, cfg.VOCAB_SIZE)
    except Exception as exc:
        return False, str(exc)
    if float(pi.min()) < cfg.MIN_STATIONARY_PROB:
        return False, f"min stationary probability {pi.min():.4g} is too small"
    symbol_probs = pi @ O
    if float(symbol_probs.min()) < cfg.MIN_SYMBOL_PROB:
        return False, f"min symbol probability {symbol_probs.min():.4g} is too small"
    # Irreducibility proxy: all states communicate within K steps.
    reach = (T > 1e-12).astype(np.int64)
    accum = np.eye(T.shape[0], dtype=np.int64)
    power = np.eye(T.shape[0], dtype=np.int64)
    for _ in range(T.shape[0]):
        power = (power @ reach > 0).astype(np.int64)
        accum = ((accum + power) > 0).astype(np.int64)
    if not np.all(accum > 0):
        return False, "transition matrix is not irreducible"
    return True, "ok"


def generate_hmm(
    K: int,
    vocab_size: int,
    alpha_t: float,
    self_bias: float,
    alpha_e: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    T = np.zeros((K, K), dtype=np.float64)
    for i in range(K):
        concentration = np.full(K, alpha_t, dtype=np.float64)
        concentration[i] += self_bias
        T[i] = rng.dirichlet(concentration)
    O = np.vstack([rng.dirichlet(np.full(vocab_size, alpha_e)) for _ in range(K)])
    pi = stationary_distribution(T)
    return canonicalize_hmm(T, O, pi, vocab_size)


def generation_parameters(cfg: Config, K: int, index: int) -> Dict[str, Any]:
    grid = [
        (a_t, bias, a_e)
        for a_t in cfg.TRANSITION_ALPHA_GRID
        for bias in cfg.SELF_BIAS_GRID
        for a_e in cfg.EMISSION_ALPHA_GRID
    ]
    a_t, bias, a_e = grid[index % len(grid)]
    cycle = index // len(grid)
    seed = cfg.MASTER_SEED + 100_000 * K + 1_009 * index + 17 * cycle
    return {
        "K": int(K),
        "bank_index": int(index),
        "alpha_t": float(a_t),
        "self_bias": float(bias),
        "alpha_e": float(a_e),
        "gen_seed": int(seed),
    }


def regenerate_from_row(row: Mapping[str, Any], cfg: Config) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    return generate_hmm(
        K=int(row["K"]),
        vocab_size=cfg.VOCAB_SIZE,
        alpha_t=float(row["alpha_t"]),
        self_bias=float(row["self_bias"]),
        alpha_e=float(row["alpha_e"]),
        seed=int(row["gen_seed"]),
    )


# -----------------------------------------------------------------------------
# Exact predictive metrics from a known HMM
# -----------------------------------------------------------------------------

def context_forward_masses(
    T: np.ndarray, O: np.ndarray, pi: np.ndarray, length: int
) -> np.ndarray:
    """Rows are unnormalized filtered state masses for all words of a fixed length."""
    K, vocab = O.shape
    if length < 1:
        return pi.reshape(1, K)
    masses = pi[None, :] * O.T  # (V, K)
    for _ in range(1, length):
        propagated = masses @ T
        masses = (propagated[:, None, :] * O.T[None, :, :]).reshape(-1, K)
    if masses.shape[0] != vocab**length:
        raise AssertionError("Unexpected number of context rows")
    return masses


def future_probability_vectors(T: np.ndarray, O: np.ndarray, length: int) -> np.ndarray:
    """Column j is P(future word j | current hidden state after the context)."""
    K, vocab = O.shape
    if length < 1:
        return np.ones((K, 1), dtype=np.float64)
    vectors = T @ O  # K x V, one future symbol
    for _ in range(1, length):
        old = vectors
        blocks = []
        # Prepend the new first symbol so lexicographic order remains stable.
        for symbol in range(vocab):
            blocks.append(T @ (O[:, symbol, None] * old))
        vectors = np.concatenate(blocks, axis=1)
    if vectors.shape[1] != vocab**length:
        raise AssertionError("Unexpected number of future columns")
    return vectors


def bayes_ce_from_context_masses(A: np.ndarray, B1: np.ndarray, eps: float = 1e-15) -> float:
    joint = A @ B1
    p_context = joint.sum(axis=1, keepdims=True)
    conditional = joint / np.clip(p_context, eps, None)
    mask = joint > 0
    return float(-np.sum(joint[mask] * np.log(np.clip(conditional[mask], eps, 1.0))))


def low_rank_centered_singular_values(
    A: np.ndarray,
    B: np.ndarray,
    pi: np.ndarray,
    tol: float = 1e-14,
) -> np.ndarray:
    """
    Singular values of the normalized centered past-future matrix without building it.

    C = D_u^{-1/2}(P - p_u p_v^T)D_v^{-1/2}
      = X Y^T, with only K+1 columns in X and Y.
    """
    p_u = A.sum(axis=1)
    p_v = pi @ B
    row_keep = p_u > tol
    col_keep = p_v > tol
    A = A[row_keep]
    p_u = p_u[row_keep]
    B = B[:, col_keep]
    p_v = p_v[col_keep]

    X1 = A / np.sqrt(p_u)[:, None]
    Y1 = (B / np.sqrt(p_v)[None, :]).T
    X = np.column_stack([X1, -np.sqrt(p_u)])
    Y = np.column_stack([Y1, np.sqrt(p_v)])

    Qx, Rx = np.linalg.qr(X, mode="reduced")
    Qy, Ry = np.linalg.qr(Y, mode="reduced")
    del Qx, Qy
    small = Rx @ Ry.T
    singular = np.linalg.svd(small, compute_uv=False)
    singular = np.sort(np.clip(singular, 0.0, None))[::-1]
    if singular.size and singular[0] > 0:
        singular = singular[singular > max(tol, singular[0] * 1e-12)]
    return singular


def spectral_summary(
    singular: np.ndarray,
    energy_levels: Sequence[float],
    tol: float,
    prefix: str,
) -> Dict[str, float]:
    singular = np.asarray(singular, dtype=np.float64)
    output: Dict[str, float] = {}
    for idx, value in enumerate(singular, start=1):
        output[f"{prefix}_sv{idx}"] = float(value)
    if singular.size == 0 or np.all(singular <= 0):
        output.update(
            {
                f"{prefix}_signal": 0.0,
                f"{prefix}_effective_rank": 0.0,
                f"{prefix}_numerical_rank": 0,
            }
        )
        for level in energy_levels:
            tag = int(round(level * 100))
            output[f"{prefix}_r{tag}"] = 0
            output[f"{prefix}_sigma{tag}"] = 0.0
            output[f"{prefix}_kappa{tag}"] = float("inf")
        return output

    energy = singular**2
    total = float(energy.sum())
    weights = energy / total
    entropy = float(-np.sum(weights * np.log(np.clip(weights, 1e-15, None))))
    output[f"{prefix}_signal"] = total
    output[f"{prefix}_effective_rank"] = float(np.exp(entropy))
    output[f"{prefix}_numerical_rank"] = int(np.sum(singular > singular[0] * tol))
    cumulative = np.cumsum(energy) / total
    for level in energy_levels:
        tag = int(round(level * 100))
        rank = int(np.searchsorted(cumulative, level, side="left") + 1)
        rank = min(rank, singular.size)
        sigma = float(singular[rank - 1])
        output[f"{prefix}_r{tag}"] = rank
        output[f"{prefix}_sigma{tag}"] = sigma
        output[f"{prefix}_kappa{tag}"] = float(singular[0] / max(sigma, 1e-15))
    return output


def observation_mixing_proxy(
    T: np.ndarray, O: np.ndarray, pi: np.ndarray, max_lag: int
) -> Tuple[float, float, float]:
    """Return hidden-chain tau, observed dependence tau proxy, and |lambda_2|."""
    eigenvalues = np.linalg.eigvals(T)
    abs_values = np.sort(np.abs(eigenvalues))[::-1]
    lambda2 = float(abs_values[1]) if len(abs_values) > 1 else 0.0
    hidden_tau = float(1.0 / max(1e-8, 1.0 - min(lambda2, 1.0 - 1e-12)))

    p = pi @ O
    joint0 = O.T @ (pi[:, None] * O)
    cov0 = joint0 - np.outer(p, p)
    denom = float(np.linalg.norm(cov0, ord="fro"))
    if denom <= 1e-15:
        return hidden_tau, 1.0, lambda2
    T_power = np.eye(T.shape[0])
    rho_sum = 0.0
    for _lag in range(1, max_lag + 1):
        T_power = T_power @ T
        joint = O.T @ ((pi[:, None] * T_power) @ O)
        cov = joint - np.outer(p, p)
        rho = min(1.0, float(np.linalg.norm(cov, ord="fro") / denom))
        rho_sum += max(0.0, rho)
    obs_tau = 1.0 + 2.0 * rho_sum
    return hidden_tau, float(obs_tau), lambda2


def compute_hmm_metrics(
    T: np.ndarray,
    O: np.ndarray,
    pi: np.ndarray,
    cfg: Config,
    future_lengths: Sequence[int],
) -> Dict[str, Any]:
    T, O, pi = canonicalize_hmm(T, O, pi, cfg.VOCAB_SIZE)
    metrics: Dict[str, Any] = {
        "K": int(T.shape[0]),
        "vocab_size": int(O.shape[1]),
        "min_stationary_prob": float(pi.min()),
        "min_symbol_prob": float((pi @ O).min()),
    }
    A_by_h: Dict[int, np.ndarray] = {}
    B1 = future_probability_vectors(T, O, 1)
    bayes_ce: Dict[int, float] = {}
    for h in range(1, cfg.MAX_CONTEXT + 1):
        A_h = context_forward_masses(T, O, pi, h)
        A_by_h[h] = A_h
        bayes_ce[h] = bayes_ce_from_context_masses(A_h, B1)
        metrics[f"bayes_ce_h{h}"] = bayes_ce[h]
    H = cfg.MAX_CONTEXT
    metrics["bayes_gap_1_to_H"] = float(bayes_ce[1] - bayes_ce[H])
    metrics["context_gain_area"] = float(
        sum(bayes_ce[h] - bayes_ce[H] for h in range(1, H))
    )

    A = A_by_h[H]
    for m in sorted(set(int(x) for x in future_lengths)):
        B = future_probability_vectors(T, O, m)
        singular = low_rank_centered_singular_values(A, B, pi)
        metrics.update(
            spectral_summary(
                singular,
                cfg.ENERGY_LEVELS,
                cfg.SPECTRAL_TOL,
                prefix=f"pred_h{H}_m{m}",
            )
        )

    hidden_tau, obs_tau, lambda2 = observation_mixing_proxy(
        T, O, pi, cfg.OBS_MIX_MAX_LAG
    )
    metrics["lambda2_abs"] = lambda2
    metrics["tau_hidden"] = min(hidden_tau, cfg.MAX_HIDDEN_TAU)
    metrics["tau_observed_proxy"] = float(obs_tau)
    return metrics


# -----------------------------------------------------------------------------
# Existing bundle import
# -----------------------------------------------------------------------------

def recursive_find_array(mapping: Mapping[str, Any], aliases: Sequence[str]) -> Optional[np.ndarray]:
    lower_aliases = {alias.lower() for alias in aliases}
    for key, value in mapping.items():
        if str(key).lower() in lower_aliases:
            try:
                return np.asarray(value, dtype=np.float64)
            except Exception:
                pass
    for value in mapping.values():
        if isinstance(value, Mapping):
            found = recursive_find_array(value, aliases)
            if found is not None:
                return found
    return None


def load_hmm_file(path: Path, cfg: Config) -> Optional[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    suffix = path.suffix.lower()
    payload: Any
    try:
        if suffix == ".npz":
            with np.load(path, allow_pickle=True) as data:
                payload = {key: data[key] for key in data.files}
        elif suffix in {".pkl", ".pickle"}:
            with path.open("rb") as handle:
                payload = pickle.load(handle)
        elif suffix == ".json":
            with path.open("r", encoding="utf-8") as handle:
                payload = json.load(handle)
        elif suffix in {".pt", ".pth"}:
            payload = safe_torch_load(path, map_location="cpu")
        else:
            return None
    except Exception:
        return None
    if not isinstance(payload, Mapping):
        return None
    T = recursive_find_array(payload, ("T", "transition", "transition_matrix", "A"))
    O = recursive_find_array(payload, ("O", "emission", "emission_matrix", "B"))
    pi = recursive_find_array(payload, ("pi", "initial", "initial_distribution", "stationary"))
    if T is None or O is None:
        return None
    try:
        return canonicalize_hmm(T, O, pi, cfg.VOCAB_SIZE)
    except Exception:
        return None


def sanitize_identifier(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    return text.strip("_.-")[:120] or "hmm"


def import_existing_hmms(
    cfg: Config, paths: Dict[str, Path], logger: logging.Logger
) -> pd.DataFrame:
    output_csv = paths["imported"] / "imported_hmm_metrics.csv"
    if output_csv.exists() and not cfg.FORCE_RECOMPUTE_METRICS:
        logger.info("Existing-HMM metrics already exist; reusing %s", output_csv)
        return pd.read_csv(output_csv)
    source = Path(cfg.EXISTING_HMM_DIR)
    if not cfg.IMPORT_EXISTING_HMMS or not source.exists():
        logger.info("Existing HMM directory is unavailable; skipping import: %s", source)
        return pd.DataFrame()
    rows: List[Dict[str, Any]] = []
    files = [
        path
        for path in source.rglob("*")
        if path.is_file() and path.suffix.lower() in {".npz", ".pkl", ".pickle", ".json", ".pt", ".pth"}
    ]
    logger.info("Scanning %d existing bundle files", len(files))
    for file_path in files:
        loaded = load_hmm_file(file_path, cfg)
        if loaded is None:
            continue
        T, O, pi = loaded
        hmm_id = "existing__" + sanitize_identifier(file_path.stem)
        target = paths["imported"] / f"{hmm_id}.npz"
        atomic_save_npz(target, T=T, O=O, pi=pi)
        metrics = compute_hmm_metrics(T, O, pi, cfg, cfg.BANK_FUTURE_LENGTHS)
        row = {
            "hmm_id": hmm_id,
            "source": "existing",
            "source_file": str(file_path),
            **metrics,
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        atomic_write_csv(output_csv, df)
    logger.info("Imported %d canonical HMMs", len(df))
    return df


# -----------------------------------------------------------------------------
# Resumable candidate bank generation
# -----------------------------------------------------------------------------

def generate_candidate_bank(
    cfg: Config, paths: Dict[str, Path], logger: logging.Logger
) -> pd.DataFrame:
    bank_csv = paths["bank"] / "candidate_bank_metrics.csv"
    if bank_csv.exists():
        bank_df = pd.read_csv(bank_csv)
    else:
        bank_df = pd.DataFrame()
    if not cfg.GENERATE_BANK:
        return bank_df

    existing_ids = set(bank_df.get("hmm_id", pd.Series(dtype=str)).astype(str).tolist())
    rows = bank_df.to_dict("records") if not bank_df.empty else []
    accepted_counts = {
        int(K): int(np.sum(bank_df.get("K", pd.Series(dtype=int)) == K))
        for K in cfg.K_VALUES
    }

    for K in cfg.K_VALUES:
        logger.info(
            "Candidate bank K=%d: %d/%d valid candidates already saved",
            K,
            accepted_counts.get(K, 0),
            cfg.BANK_PER_K,
        )
        index = 0
        attempts = 0
        max_attempts = max(cfg.BANK_PER_K * 20, cfg.BANK_PER_K + 100)
        while accepted_counts.get(K, 0) < cfg.BANK_PER_K and attempts < max_attempts:
            params = generation_parameters(cfg, K, index)
            hmm_id = f"bank_K{K}_i{index:06d}"
            index += 1
            attempts += 1
            if hmm_id in existing_ids:
                continue
            try:
                T, O, pi = generate_hmm(
                    K=K,
                    vocab_size=cfg.VOCAB_SIZE,
                    alpha_t=params["alpha_t"],
                    self_bias=params["self_bias"],
                    alpha_e=params["alpha_e"],
                    seed=params["gen_seed"],
                )
                valid, reason = validate_hmm(T, O, pi, cfg)
                if not valid:
                    continue
                metrics = compute_hmm_metrics(
                    T, O, pi, cfg, cfg.BANK_FUTURE_LENGTHS
                )
                if metrics["context_gain_area"] < cfg.MIN_CONTEXT_GAIN:
                    continue
                row = {
                    "hmm_id": hmm_id,
                    "source": "bank",
                    **params,
                    **metrics,
                }
                rows.append(row)
                existing_ids.add(hmm_id)
                accepted_counts[K] = accepted_counts.get(K, 0) + 1
                if accepted_counts[K] % cfg.BANK_SAVE_EVERY == 0:
                    current = pd.DataFrame(rows)
                    if not current.empty:
                        current = current.sort_values(["K", "hmm_id"])
                        atomic_write_csv(bank_csv, current)
                    logger.info(
                        "Saved candidate bank checkpoint: K=%d, accepted=%d",
                        K,
                        accepted_counts[K],
                    )
            except KeyboardInterrupt:
                current = pd.DataFrame(rows)
                if not current.empty:
                    current = current.sort_values(["K", "hmm_id"])
                    atomic_write_csv(bank_csv, current)
                raise
            except Exception as exc:
                if not cfg.CONTINUE_AFTER_ERROR:
                    raise
                record_failure(paths, "candidate_bank", hmm_id, exc)
                continue
        if accepted_counts.get(K, 0) < cfg.BANK_PER_K:
            logger.warning(
                "K=%d stopped with %d valid candidates after %d attempts",
                K,
                accepted_counts.get(K, 0),
                attempts,
            )
        current = pd.DataFrame(rows)
        if not current.empty:
            current = current.sort_values(["K", "hmm_id"])
            atomic_write_csv(bank_csv, current)
    bank_df = pd.DataFrame(rows)
    if not bank_df.empty:
        bank_df = bank_df.sort_values(["K", "hmm_id"]).reset_index(drop=True)
    logger.info("Candidate bank contains %d HMMs", len(bank_df))
    return bank_df


# -----------------------------------------------------------------------------
# Matched-pair selection
# -----------------------------------------------------------------------------

def robust_standardize(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    output = frame.copy()
    for col in columns:
        values = pd.to_numeric(output[col], errors="coerce").astype(float)
        median = float(values.median())
        mad = float(np.median(np.abs(values - median)))
        scale = max(1e-8, 1.4826 * mad)
        output[f"z__{col}"] = (values - median) / scale
    return output


def greedy_pairs(candidates: List[Tuple[float, str, str, Dict[str, Any]]], limit: int) -> List[Dict[str, Any]]:
    used: set[str] = set()
    selected: List[Dict[str, Any]] = []
    for score, a, b, info in sorted(candidates, key=lambda item: item[0], reverse=True):
        if len(selected) >= limit:
            break
        if a in used or b in used:
            continue
        used.add(a)
        used.add(b)
        selected.append({"hmm_a": a, "hmm_b": b, "selection_score": score, **info})
    return selected


def select_matched_pairs(
    all_metrics: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Select strictly controlled same-K pairs and cross-K pairs.

    v4 changes:
      * raw ratio constraints prevent large gain/sigma/mixing mismatches;
      * rank contrast no longer rewards rank difference at any cost;
      * sigma contrast requires a large sigma ratio while matching gain/rank/mixing;
      * cross-K pairs are formed *within the already selected HMM set*, so they
        do not get dropped by the 24-HMM cap;
      * all stored pair metrics are copied from the exact table used to select.
    """
    selected_csv = paths["selected"] / "selected_hmms.csv"
    pairs_csv = paths["selected"] / "matched_pairs.csv"
    if selected_csv.exists() and pairs_csv.exists() and not cfg.FORCE_RESELECT:
        logger.info("Matched-pair selection already exists; reusing it")
        return pd.read_csv(selected_csv), pd.read_csv(pairs_csv)

    m = cfg.SELECTION_FUTURE_LENGTH
    rank_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_r95"
    sigma_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_sigma95"
    required = [
        "hmm_id", "K", "context_gain_area", rank_col, sigma_col,
        "tau_observed_proxy",
    ]
    missing = [col for col in required if col not in all_metrics.columns]
    if missing:
        raise ValueError(f"Missing columns needed for matching: {missing}")

    df = all_metrics.copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=required)
    df = df[df["context_gain_area"] > cfg.MIN_CONTEXT_GAIN].copy()
    df[rank_col] = df[rank_col].astype(int)
    for col in ("context_gain_area", sigma_col, "tau_observed_proxy"):
        df[col] = df[col].astype(float)

    def ratio(x: float, y: float) -> float:
        lo = max(min(float(x), float(y)), 1e-15)
        return max(float(x), float(y)) / lo

    def control_log_ratios(gr: float, sr: float, tr: float) -> float:
        return float(math.sqrt(math.log(gr) ** 2 + 0.5 * math.log(sr) ** 2 + 0.5 * math.log(tr) ** 2))

    def take_disjoint(candidates: List[Tuple[float, str, str, Dict[str, Any]]], limit: int,
                      globally_used: set[str]) -> List[Dict[str, Any]]:
        local_used: set[str] = set()
        chosen: List[Dict[str, Any]] = []
        for score, a, b, info in sorted(candidates, key=lambda x: x[0], reverse=True):
            if len(chosen) >= limit:
                break
            if a in globally_used or b in globally_used or a in local_used or b in local_used:
                continue
            local_used.update((a, b))
            chosen.append({"hmm_a": a, "hmm_b": b, "selection_score": float(score), **info})
        globally_used.update(local_used)
        return chosen

    pair_rows: List[Dict[str, Any]] = []
    same_k_selected: List[str] = []
    globally_used: set[str] = set()

    # A. Same-K rank contrasts with strict matching on gain, sigma and mixing.
    for K, group in df.groupby("K"):
        records = group.to_dict("records")
        candidates: List[Tuple[float, str, str, Dict[str, Any]]] = []
        for i in range(len(records)):
            for j in range(i + 1, len(records)):
                a, b = records[i], records[j]
                rank_diff = abs(int(a[rank_col]) - int(b[rank_col]))
                if rank_diff < cfg.MIN_RANK_CONTRAST:
                    continue
                gr = ratio(a["context_gain_area"], b["context_gain_area"])
                sr = ratio(a[sigma_col], b[sigma_col])
                tr = ratio(a["tau_observed_proxy"], b["tau_observed_proxy"])
                if (gr > cfg.RANK_MAX_GAIN_RATIO or sr > cfg.RANK_MAX_SIGMA_RATIO
                        or tr > cfg.RANK_MAX_TAU_RATIO):
                    continue
                control = control_log_ratios(gr, sr, tr)
                # Rank difference matters, but only after strict raw-ratio controls.
                score = 10.0 * rank_diff - control
                easier, harder = (a, b) if int(a[rank_col]) < int(b[rank_col]) else (b, a)
                candidates.append((score, str(easier["hmm_id"]), str(harder["hmm_id"]), {
                    "pair_type": "rank_contrast", "K_a": int(easier["K"]), "K_b": int(harder["K"]),
                    "expected_harder": str(harder["hmm_id"]), "control_distance": control,
                    "rank_a": int(easier[rank_col]), "rank_b": int(harder[rank_col]),
                    "sigma_a": float(easier[sigma_col]), "sigma_b": float(harder[sigma_col]),
                    "gain_a": float(easier["context_gain_area"]), "gain_b": float(harder["context_gain_area"]),
                    "tau_a": float(easier["tau_observed_proxy"]), "tau_b": float(harder["tau_observed_proxy"]),
                    "gain_ratio": gr, "sigma_ratio": sr, "tau_ratio": tr,
                    "match_protocol": cfg.MATCH_PROTOCOL_VERSION,
                }))
        chosen = take_disjoint(candidates, cfg.RANK_PAIRS_PER_K, globally_used)
        if len(chosen) < cfg.RANK_PAIRS_PER_K:
            raise RuntimeError(f"K={K}: only {len(chosen)} strict rank pairs found; expected {cfg.RANK_PAIRS_PER_K}")
        pair_rows.extend(chosen)
        for p in chosen:
            same_k_selected.extend((p["hmm_a"], p["hmm_b"]))

    # B. Same-K sigma contrasts, disjoint from rank-pair HMMs.
    for K, group in df.groupby("K"):
        records = group.to_dict("records")
        candidates = []
        for i in range(len(records)):
            for j in range(i + 1, len(records)):
                a, b = records[i], records[j]
                if int(a[rank_col]) != int(b[rank_col]):
                    continue
                gr = ratio(a["context_gain_area"], b["context_gain_area"])
                sr = ratio(a[sigma_col], b[sigma_col])
                tr = ratio(a["tau_observed_proxy"], b["tau_observed_proxy"])
                if sr < cfg.SIGMA_MIN_RATIO or gr > cfg.SIGMA_MAX_GAIN_RATIO or tr > cfg.SIGMA_MAX_TAU_RATIO:
                    continue
                control = float(math.sqrt(math.log(gr) ** 2 + math.log(tr) ** 2))
                score = 3.0 * math.log(sr) - control
                easier, harder = (a, b) if float(a[sigma_col]) > float(b[sigma_col]) else (b, a)
                candidates.append((score, str(easier["hmm_id"]), str(harder["hmm_id"]), {
                    "pair_type": "sigma_contrast", "K_a": int(easier["K"]), "K_b": int(harder["K"]),
                    "expected_harder": str(harder["hmm_id"]), "control_distance": control,
                    "rank_a": int(easier[rank_col]), "rank_b": int(harder[rank_col]),
                    "sigma_a": float(easier[sigma_col]), "sigma_b": float(harder[sigma_col]),
                    "gain_a": float(easier["context_gain_area"]), "gain_b": float(harder["context_gain_area"]),
                    "tau_a": float(easier["tau_observed_proxy"]), "tau_b": float(harder["tau_observed_proxy"]),
                    "gain_ratio": gr, "sigma_ratio": sr, "tau_ratio": tr,
                    "match_protocol": cfg.MATCH_PROTOCOL_VERSION,
                }))
        chosen = take_disjoint(candidates, cfg.SIGMA_PAIRS_PER_K, globally_used)
        if len(chosen) < cfg.SIGMA_PAIRS_PER_K:
            raise RuntimeError(f"K={K}: only {len(chosen)} strict sigma pairs found; expected {cfg.SIGMA_PAIRS_PER_K}")
        pair_rows.extend(chosen)
        for p in chosen:
            same_k_selected.extend((p["hmm_a"], p["hmm_b"]))

    retained_ids = list(dict.fromkeys(same_k_selected))
    if len(retained_ids) > cfg.MAX_SELECTED_HMMS:
        raise RuntimeError(f"Strict same-K selection produced {len(retained_ids)} HMMs, exceeding cap {cfg.MAX_SELECTED_HMMS}")

    # C. Cross-K profile matches drawn only from the retained same-K HMM set.
    selected_metric_df = df[df["hmm_id"].astype(str).isin(retained_ids)].copy()
    records = selected_metric_df.to_dict("records")
    cross_candidates: List[Tuple[float, str, str, Dict[str, Any]]] = []
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            a, b = records[i], records[j]
            if int(a["K"]) == int(b["K"]):
                continue
            gr = ratio(a["context_gain_area"], b["context_gain_area"])
            sr = ratio(a[sigma_col], b[sigma_col])
            tr = ratio(a["tau_observed_proxy"], b["tau_observed_proxy"])
            rank_diff = abs(int(a[rank_col]) - int(b[rank_col]))
            distance = float(math.sqrt(math.log(gr) ** 2 + rank_diff ** 2 + math.log(sr) ** 2 + math.log(tr) ** 2))
            cross_candidates.append((-distance, str(a["hmm_id"]), str(b["hmm_id"]), {
                "pair_type": "cross_K_matched", "K_a": int(a["K"]), "K_b": int(b["K"]),
                "expected_harder": "", "control_distance": distance,
                "rank_a": int(a[rank_col]), "rank_b": int(b[rank_col]),
                "sigma_a": float(a[sigma_col]), "sigma_b": float(b[sigma_col]),
                "gain_a": float(a["context_gain_area"]), "gain_b": float(b["context_gain_area"]),
                "tau_a": float(a["tau_observed_proxy"]), "tau_b": float(b["tau_observed_proxy"]),
                "gain_ratio": gr, "sigma_ratio": sr, "tau_ratio": tr,
                "match_protocol": cfg.MATCH_PROTOCOL_VERSION,
            }))
    # Cross-K pairing has its own usage set; HMMs are intentionally reused from same-K contrasts.
    cross_used: set[str] = set()
    cross_chosen: List[Dict[str, Any]] = []
    for score, a, b, info in sorted(cross_candidates, key=lambda x: x[0], reverse=True):
        if len(cross_chosen) >= cfg.CROSS_K_PAIRS:
            break
        if a in cross_used or b in cross_used:
            continue
        cross_used.update((a, b))
        cross_chosen.append({"hmm_a": a, "hmm_b": b, "selection_score": float(score), **info})
    pair_rows.extend(cross_chosen)

    selected_df = all_metrics[all_metrics["hmm_id"].astype(str).isin(retained_ids)].copy()
    selected_df["selection_order"] = selected_df["hmm_id"].astype(str).map(
        {hmm_id: idx for idx, hmm_id in enumerate(retained_ids)}
    )
    selected_df["match_protocol"] = cfg.MATCH_PROTOCOL_VERSION
    selected_df = selected_df.sort_values("selection_order")
    pairs_df = pd.DataFrame(pair_rows)
    if not pairs_df.empty:
        pairs_df.insert(0, "pair_id", [f"pair_{i:02d}" for i in range(len(pairs_df))])
    atomic_write_csv(selected_csv, selected_df)
    atomic_write_csv(pairs_csv, pairs_df)
    logger.info("Selected %d HMMs: %d same-K pairs plus %d cross-K pairs", len(selected_df),
                sum(pairs_df["pair_type"] != "cross_K_matched") if not pairs_df.empty else 0,
                sum(pairs_df["pair_type"] == "cross_K_matched") if not pairs_df.empty else 0)
    return selected_df, pairs_df

def materialize_selected_hmms(
    selected_df: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> pd.DataFrame:
    full_metrics_path = paths["selected"] / "selected_hmm_metrics_full.csv"
    rows: List[Dict[str, Any]] = []
    bank_lookup = selected_df.set_index("hmm_id", drop=False).to_dict("index")
    for hmm_id, row in bank_lookup.items():
        target = paths["selected"] / f"{sanitize_identifier(str(hmm_id))}.npz"
        try:
            if target.exists():
                with np.load(target) as data:
                    T, O, pi = data["T"], data["O"], data["pi"]
            elif str(row.get("source", "")) == "bank":
                T, O, pi = regenerate_from_row(row, cfg)
                atomic_save_npz(target, T=T, O=O, pi=pi)
            else:
                imported = paths["imported"] / f"{sanitize_identifier(str(hmm_id))}.npz"
                if not imported.exists():
                    raise FileNotFoundError(f"Cannot locate imported HMM {hmm_id}")
                shutil.copy2(imported, target)
                with np.load(target) as data:
                    T, O, pi = data["T"], data["O"], data["pi"]
            metrics = compute_hmm_metrics(
                T, O, pi, cfg, cfg.SELECTED_FUTURE_LENGTHS
            )
            rows.append({**dict(row), **metrics, "hmm_file": str(target)})
        except Exception as exc:
            record_failure(paths, "materialize_selected", str(hmm_id), exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise
    full_df = pd.DataFrame(rows)
    atomic_write_csv(full_metrics_path, full_df)
    logger.info("Materialized %d selected HMMs", len(full_df))
    return full_df


# -----------------------------------------------------------------------------
# Sequence generation and cached datasets
# -----------------------------------------------------------------------------

def sample_categorical_rows(probabilities: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    cumulative = np.cumsum(probabilities, axis=1)
    draws = rng.random(probabilities.shape[0])[:, None]
    return (draws > cumulative).sum(axis=1).astype(np.int64)


def generate_sequences(
    T: np.ndarray,
    O: np.ndarray,
    pi: np.ndarray,
    n_sequences: int,
    length: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    K, vocab = O.shape
    states = rng.choice(K, size=n_sequences, p=pi)
    sequences = np.empty((n_sequences, length), dtype=np.int16)
    for t in range(length):
        emissions = O[states]
        sequences[:, t] = sample_categorical_rows(emissions, rng)
        transitions = T[states]
        states = sample_categorical_rows(transitions, rng)
    return sequences


def ensure_dataset(
    hmm_id: str,
    T: np.ndarray,
    O: np.ndarray,
    pi: np.ndarray,
    cfg: Config,
    paths: Dict[str, Path],
) -> Path:
    dataset_path = paths["datasets"] / f"{sanitize_identifier(hmm_id)}.npz"
    if dataset_path.exists():
        try:
            with np.load(dataset_path) as data:
                if (
                    data["train"].shape[0] >= cfg.N_TRAIN_MAX
                    and data["train"].shape[1] == cfg.SEQUENCE_LENGTH
                    and data["val"].shape[0] >= cfg.N_VAL
                    and data["test"].shape[0] >= cfg.N_TEST
                ):
                    return dataset_path
        except Exception:
            pass
    base = cfg.MASTER_SEED + stable_hash(hmm_id)
    train = generate_sequences(T, O, pi, cfg.N_TRAIN_MAX, cfg.SEQUENCE_LENGTH, base + 1)
    val = generate_sequences(T, O, pi, cfg.N_VAL, cfg.SEQUENCE_LENGTH, base + 2)
    test = generate_sequences(T, O, pi, cfg.N_TEST, cfg.SEQUENCE_LENGTH, base + 3)
    atomic_save_npz(dataset_path, train=train, val=val, test=test)
    return dataset_path


def load_dataset(path: Path) -> Dict[str, np.ndarray]:
    with np.load(path) as data:
        return {key: data[key].copy() for key in ("train", "val", "test")}


def sample_training_windows(
    sequences: np.ndarray,
    max_context: int,
    vocab_size: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n, length = sequences.shape
    h_values = rng.integers(1, max_context + 1, size=n)
    end_positions = np.array(
        [rng.integers(h, length) for h in h_values], dtype=np.int64
    )
    pad_token = vocab_size
    x = np.full((n, max_context), pad_token, dtype=np.int64)
    y = np.empty(n, dtype=np.int64)
    for idx, (h, end) in enumerate(zip(h_values, end_positions)):
        x[idx, :h] = sequences[idx, end - h : end]
        y[idx] = sequences[idx, end]
    return x, h_values.astype(np.int64), y


def fixed_eval_windows(
    sequences: np.ndarray,
    h: int,
    max_context: int,
    vocab_size: int,
    windows_per_sequence: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n, length = sequences.shape
    total = n * windows_per_sequence
    pad_token = vocab_size
    x = np.full((total, max_context), pad_token, dtype=np.int64)
    lengths = np.full(total, h, dtype=np.int64)
    y = np.empty(total, dtype=np.int64)
    row = 0
    for seq_idx in range(n):
        if windows_per_sequence >= length - h:
            ends = np.arange(h, length)
        else:
            ends = rng.choice(np.arange(h, length), size=windows_per_sequence, replace=False)
        for end in ends[:windows_per_sequence]:
            x[row, :h] = sequences[seq_idx, end - h : end]
            y[row] = sequences[seq_idx, end]
            row += 1
    return x[:row], lengths[:row], y[:row]


# -----------------------------------------------------------------------------
# Neural models
# -----------------------------------------------------------------------------

class GRUPredictor(nn.Module):
    def __init__(self, vocab_size: int, dimension: int, pad_token: int, dropout: float = 0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, dimension, padding_idx=pad_token)
        self.gru = nn.GRU(
            input_size=dimension,
            hidden_size=dimension,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(dimension, vocab_size)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x)
        packed = pack_padded_sequence(
            embedded, lengths.detach().cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        representation = self.dropout(hidden[-1])
        return self.output(representation)


class TransformerPredictor(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dimension: int,
        max_context: int,
        pad_token: int,
        nhead: int,
        layers: int,
        ff_multiplier: int,
        dropout: float,
    ):
        super().__init__()
        if dimension % nhead != 0:
            raise ValueError(f"dimension={dimension} must be divisible by nhead={nhead}")
        self.max_context = max_context
        self.pad_token = pad_token
        self.embedding = nn.Embedding(vocab_size + 1, dimension, padding_idx=pad_token)
        self.position = nn.Embedding(max_context, dimension)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dimension,
            nhead=nhead,
            dim_feedforward=max(dimension, ff_multiplier * dimension),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.norm = nn.LayerNorm(dimension)
        self.output = nn.Linear(dimension, vocab_size)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        batch, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch, -1)
        hidden = self.embedding(x) + self.position(positions)
        key_padding_mask = x.eq(self.pad_token)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1
        )
        encoded = self.encoder(
            hidden, mask=causal_mask, src_key_padding_mask=key_padding_mask
        )
        last_indices = (lengths - 1).clamp_min(0)
        representation = encoded[torch.arange(batch, device=x.device), last_indices]
        return self.output(self.norm(representation))


def build_model(architecture: str, dimension: int, cfg: Config) -> nn.Module:
    pad_token = cfg.VOCAB_SIZE
    if architecture.lower() == "gru":
        return GRUPredictor(cfg.VOCAB_SIZE, dimension, pad_token, cfg.DROPOUT)
    if architecture.lower() == "transformer":
        return TransformerPredictor(
            vocab_size=cfg.VOCAB_SIZE,
            dimension=dimension,
            max_context=cfg.MAX_CONTEXT,
            pad_token=pad_token,
            nhead=cfg.TRANSFORMER_HEADS,
            layers=cfg.TRANSFORMER_LAYERS,
            ff_multiplier=cfg.TRANSFORMER_FF_MULTIPLIER,
            dropout=cfg.DROPOUT,
        )
    raise ValueError(f"Unknown architecture: {architecture}")


def iter_minibatches(
    x: np.ndarray,
    lengths: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    seed: int,
) -> Iterable[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(y))
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start : start + batch_size]
        yield x[batch_idx], lengths[batch_idx], y[batch_idx]


def evaluate_model_ce(
    model: nn.Module,
    sequences: np.ndarray,
    cfg: Config,
    device: torch.device,
    batch_size: int,
    seed_base: int,
) -> Dict[int, float]:
    model.eval()
    results: Dict[int, float] = {}
    with torch.no_grad():
        for h in range(1, cfg.MAX_CONTEXT + 1):
            x, lengths, y = fixed_eval_windows(
                sequences,
                h=h,
                max_context=cfg.MAX_CONTEXT,
                vocab_size=cfg.VOCAB_SIZE,
                windows_per_sequence=cfg.EVAL_WINDOWS_PER_SEQUENCE,
                seed=seed_base + h,
            )
            total_loss = 0.0
            total_count = 0
            for start in range(0, len(y), batch_size):
                xb = torch.as_tensor(x[start : start + batch_size], dtype=torch.long, device=device)
                lb = torch.as_tensor(lengths[start : start + batch_size], dtype=torch.long, device=device)
                yb = torch.as_tensor(y[start : start + batch_size], dtype=torch.long, device=device)
                logits = model(xb, lb)
                loss = F.cross_entropy(logits, yb, reduction="sum")
                total_loss += float(loss.item())
                total_count += int(yb.numel())
            results[h] = total_loss / max(total_count, 1)
    return results


def validation_score(
    model: nn.Module,
    val_sequences: np.ndarray,
    cfg: Config,
    device: torch.device,
    batch_size: int,
    seed_base: int,
) -> float:
    curve = evaluate_model_ce(model, val_sequences, cfg, device, batch_size, seed_base)
    return float(np.mean(list(curve.values())))


def condition_directory(
    base: Path,
    hmm_id: str,
    architecture: str,
    dimension: int,
    seed: int,
    n_train: int,
) -> Path:
    return (
        base
        / sanitize_identifier(hmm_id)
        / architecture.lower()
        / f"d{dimension}"
        / f"n{n_train}"
        / f"seed{seed}"
    )


def train_one_condition(
    hmm_id: str,
    architecture: str,
    dimension: int,
    seed: int,
    n_train: int,
    dataset_path: Path,
    run_base: Path,
    cfg: Config,
    device: torch.device,
    logger: logging.Logger,
) -> Dict[str, Any]:
    run_dir = condition_directory(run_base, hmm_id, architecture, dimension, seed, n_train)
    run_dir.mkdir(parents=True, exist_ok=True)
    done_path = run_dir / "DONE.json"
    failed_path = run_dir / "FAILED.json"
    checkpoint_path = run_dir / "checkpoint.pt"
    history_path = run_dir / "history.csv"

    if done_path.exists() and not cfg.FORCE_RETRAIN:
        with done_path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    if failed_path.exists() and not cfg.RETRY_FAILED_CONDITIONS and not cfg.FORCE_RETRAIN:
        with failed_path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    if cfg.FORCE_RETRAIN:
        for path in (done_path, failed_path, checkpoint_path, history_path):
            if path.exists():
                path.unlink()

    data = load_dataset(dataset_path)
    train_sequences = data["train"][:n_train]
    val_sequences = data["val"][: cfg.N_VAL]
    test_sequences = data["test"][: cfg.N_TEST]
    condition_seed = cfg.MASTER_SEED + stable_hash(
        f"{hmm_id}|{architecture}|{dimension}|{seed}|{n_train}"
    )
    set_global_seed(condition_seed, cfg.DETERMINISTIC_TORCH)

    batch_size = cfg.BATCH_SIZE
    start_epoch = 1
    best_val = float("inf")
    best_epoch = 0
    no_improve = 0
    history: List[Dict[str, Any]] = []

    model = build_model(architecture, dimension, cfg).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY
    )

    if checkpoint_path.exists() and not cfg.FORCE_RETRAIN:
        checkpoint = safe_torch_load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val = float(checkpoint["best_val"])
        best_epoch = int(checkpoint["best_epoch"])
        no_improve = int(checkpoint.get("no_improve", 0))
        batch_size = int(checkpoint.get("batch_size", batch_size))
        history = list(checkpoint.get("history", []))
        logger.info(
            "Resuming %s %s d=%d seed=%d n=%d from epoch %d",
            hmm_id,
            architecture,
            dimension,
            seed,
            n_train,
            start_epoch,
        )

    try:
        for epoch in range(start_epoch, cfg.MAX_EPOCHS + 1):
            while True:
                try:
                    model.train()
                    x, lengths, y = sample_training_windows(
                        train_sequences,
                        max_context=cfg.MAX_CONTEXT,
                        vocab_size=cfg.VOCAB_SIZE,
                        seed=condition_seed + 1000 * epoch,
                    )
                    epoch_loss = 0.0
                    epoch_count = 0
                    for xb_np, lb_np, yb_np in iter_minibatches(
                        x,
                        lengths,
                        y,
                        batch_size=batch_size,
                        seed=condition_seed + 2000 * epoch,
                    ):
                        xb = torch.as_tensor(xb_np, dtype=torch.long, device=device)
                        lb = torch.as_tensor(lb_np, dtype=torch.long, device=device)
                        yb = torch.as_tensor(yb_np, dtype=torch.long, device=device)
                        optimizer.zero_grad(set_to_none=True)
                        logits = model(xb, lb)
                        loss = F.cross_entropy(logits, yb)
                        if not torch.isfinite(loss):
                            raise FloatingPointError(f"Non-finite training loss: {loss.item()}")
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                        optimizer.step()
                        epoch_loss += float(loss.item()) * len(yb_np)
                        epoch_count += len(yb_np)
                    train_ce = epoch_loss / max(epoch_count, 1)
                    val_ce = validation_score(
                        model,
                        val_sequences,
                        cfg,
                        device,
                        batch_size=max(batch_size, 32),
                        seed_base=condition_seed + 300000,
                    )
                    break
                except RuntimeError as exc:
                    if "out of memory" in str(exc).lower() and batch_size > cfg.MIN_BATCH_SIZE:
                        old = batch_size
                        batch_size = max(cfg.MIN_BATCH_SIZE, batch_size // 2)
                        logger.warning(
                            "CUDA OOM for %s/%s/d%d/seed%d; batch %d -> %d",
                            hmm_id,
                            architecture,
                            dimension,
                            seed,
                            old,
                            batch_size,
                        )
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()
                        continue
                    raise

            improved = val_ce < best_val - 1e-8
            if improved:
                best_val = val_ce
                best_epoch = epoch
                no_improve = 0
                best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
                atomic_torch_save(run_dir / "best_model.pt", best_state)
            else:
                no_improve += 1

            history.append(
                {
                    "epoch": epoch,
                    "train_ce": train_ce,
                    "val_ce": val_ce,
                    "best_val": best_val,
                    "best_epoch": best_epoch,
                    "batch_size": batch_size,
                }
            )
            atomic_write_csv(history_path, pd.DataFrame(history))
            if cfg.CHECKPOINT_EVERY_EPOCH:
                atomic_torch_save(
                    checkpoint_path,
                    {
                        "epoch": epoch,
                        "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "no_improve": no_improve,
                        "batch_size": batch_size,
                        "history": history,
                        "condition_seed": condition_seed,
                    },
                )

            logger.info(
                "%s | %s d=%d seed=%d n=%d | epoch %d | train %.4f | val %.4f",
                hmm_id,
                architecture,
                dimension,
                seed,
                n_train,
                epoch,
                train_ce,
                val_ce,
            )
            if epoch >= cfg.MIN_EPOCHS and no_improve >= cfg.EARLY_STOP_PATIENCE:
                break

        best_state_path = run_dir / "best_model.pt"
        if best_state_path.exists():
            model.load_state_dict(safe_torch_load(best_state_path, map_location=device))
        test_curve = evaluate_model_ce(
            model,
            test_sequences,
            cfg,
            device,
            batch_size=max(batch_size, 32),
            seed_base=condition_seed + 400000,
        )
        result: Dict[str, Any] = {
            "status": "done",
            "hmm_id": hmm_id,
            "architecture": architecture,
            "dimension": int(dimension),
            "seed": int(seed),
            "n_train": int(n_train),
            "best_epoch": int(best_epoch),
            "best_val_ce": float(best_val),
            "final_batch_size": int(batch_size),
            "epochs_completed": int(history[-1]["epoch"] if history else 0),
            "parameter_count": int(sum(p.numel() for p in model.parameters())),
            "device": str(device),
        }
        for h, value in test_curve.items():
            result[f"test_ce_h{h}"] = float(value)
        atomic_write_json(done_path, result)
        if failed_path.exists():
            failed_path.unlink()
        return result
    except KeyboardInterrupt:
        logger.warning("Interrupted during %s; checkpoint is preserved", run_dir)
        raise
    except Exception as exc:
        failure = {
            "status": "failed",
            "hmm_id": hmm_id,
            "architecture": architecture,
            "dimension": int(dimension),
            "seed": int(seed),
            "n_train": int(n_train),
            "exception_type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        }
        atomic_write_json(failed_path, failure)
        raise
    finally:
        del model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()


def prepare_all_datasets(
    selected_full: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> Dict[str, Path]:
    mapping: Dict[str, Path] = {}
    for _, row in selected_full.iterrows():
        hmm_id = str(row["hmm_id"])
        try:
            with np.load(Path(row["hmm_file"])) as data:
                T, O, pi = data["T"], data["O"], data["pi"]
            mapping[hmm_id] = ensure_dataset(hmm_id, T, O, pi, cfg, paths)
        except Exception as exc:
            record_failure(paths, "dataset", hmm_id, exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise
    logger.info("Prepared %d cached sequence datasets", len(mapping))
    return mapping


def run_training_grid(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
    device: torch.device,
) -> None:
    if not cfg.RUN_TRAINING:
        return
    conditions = [
        (str(row["hmm_id"]), arch, int(dim), int(seed))
        for _, row in selected_full.iterrows()
        for arch in cfg.ARCHITECTURES
        for dim in cfg.DIMENSIONS
        for seed in cfg.SEEDS
    ]
    logger.info("Main training grid contains %d conditions", len(conditions))
    for index, (hmm_id, arch, dim, seed) in enumerate(conditions, start=1):
        if hmm_id not in dataset_paths:
            continue
        try:
            train_one_condition(
                hmm_id=hmm_id,
                architecture=arch,
                dimension=dim,
                seed=seed,
                n_train=cfg.N_TRAIN_MAIN,
                dataset_path=dataset_paths[hmm_id],
                run_base=paths["runs"],
                cfg=cfg,
                device=device,
                logger=logger,
            )
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            record_failure(paths, "training", f"{hmm_id}_{arch}_d{dim}_s{seed}", exc)
            logger.error(
                "Condition failed (%d/%d): %s %s d=%d seed=%d | %s",
                index,
                len(conditions),
                hmm_id,
                arch,
                dim,
                seed,
                exc,
            )
            if not cfg.CONTINUE_AFTER_ERROR:
                raise


def choose_sample_size_hmms(
    pairs_df: pd.DataFrame, selected_full: pd.DataFrame, cfg: Config
) -> List[str]:
    ids: List[str] = []
    if not pairs_df.empty:
        for _, pair in pairs_df.iterrows():
            for column in ("hmm_a", "hmm_b"):
                hmm_id = str(pair[column])
                if hmm_id not in ids:
                    ids.append(hmm_id)
                if len(ids) >= cfg.SAMPLE_SIZE_MAX_HMMS:
                    return ids
    for hmm_id in selected_full["hmm_id"].astype(str):
        if hmm_id not in ids:
            ids.append(hmm_id)
        if len(ids) >= cfg.SAMPLE_SIZE_MAX_HMMS:
            break
    return ids


def run_sample_size_grid(
    sample_hmms: Sequence[str],
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
    device: torch.device,
) -> None:
    if not cfg.RUN_SAMPLE_SIZE_EXPERIMENT or not cfg.RUN_TRAINING:
        return
    conditions = [
        (hmm_id, arch, int(n), int(seed))
        for hmm_id in sample_hmms
        for arch in cfg.ARCHITECTURES
        for n in cfg.SAMPLE_SIZE_VALUES
        for seed in cfg.SAMPLE_SIZE_SEEDS
    ]
    logger.info("Sample-size grid contains %d conditions", len(conditions))
    for hmm_id, arch, n_train, seed in conditions:
        if hmm_id not in dataset_paths:
            continue
        try:
            train_one_condition(
                hmm_id=hmm_id,
                architecture=arch,
                dimension=cfg.SAMPLE_SIZE_DIMENSION,
                seed=seed,
                n_train=n_train,
                dataset_path=dataset_paths[hmm_id],
                run_base=paths["sample_runs"],
                cfg=cfg,
                device=device,
                logger=logger,
            )
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            record_failure(paths, "sample_size_training", f"{hmm_id}_{arch}_n{n_train}_s{seed}", exc)
            logger.error("Sample-size condition failed: %s", exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise


# -----------------------------------------------------------------------------
# Empirical predictive rank from observed sequences
# -----------------------------------------------------------------------------

def word_indices(blocks: np.ndarray, vocab_size: int) -> np.ndarray:
    indices = np.zeros(blocks.shape[0], dtype=np.int64)
    for column in range(blocks.shape[1]):
        indices = indices * vocab_size + blocks[:, column]
    return indices


def empirical_joint_matrix(
    sequences: np.ndarray,
    h: int,
    m: int,
    vocab_size: int,
    max_windows: int,
    seed: int,
) -> np.ndarray:
    contexts: List[np.ndarray] = []
    futures: List[np.ndarray] = []
    for sequence in sequences:
        for end in range(h, len(sequence) - m + 1):
            contexts.append(sequence[end - h : end])
            futures.append(sequence[end : end + m])
    if not contexts:
        raise ValueError("No windows available for empirical rank")
    context_array = np.asarray(contexts, dtype=np.int64)
    future_array = np.asarray(futures, dtype=np.int64)
    if len(context_array) > max_windows:
        rng = np.random.default_rng(seed)
        indices = rng.choice(len(context_array), size=max_windows, replace=False)
        context_array = context_array[indices]
        future_array = future_array[indices]
    u = word_indices(context_array, vocab_size)
    v = word_indices(future_array, vocab_size)
    n_u = vocab_size**h
    n_v = vocab_size**m
    counts = np.bincount(u * n_v + v, minlength=n_u * n_v).reshape(n_u, n_v)
    return counts.astype(np.float64) / max(counts.sum(), 1.0)


def singular_values_from_joint(P: np.ndarray) -> np.ndarray:
    p_u = P.sum(axis=1)
    p_v = P.sum(axis=0)
    row_keep = p_u > 0
    col_keep = p_v > 0
    P = P[np.ix_(row_keep, col_keep)]
    p_u = p_u[row_keep]
    p_v = p_v[col_keep]
    centered = P - np.outer(p_u, p_v)
    normalized = centered / np.sqrt(np.outer(p_u, p_v))
    return np.linalg.svd(normalized, compute_uv=False)


def empirical_rank_analysis(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> pd.DataFrame:
    output = paths["analysis"] / "empirical_predictive_rank.csv"
    if output.exists() and not cfg.FORCE_RECOMPUTE_METRICS:
        return pd.read_csv(output)
    if not cfg.RUN_EMPIRICAL_RANK:
        return pd.DataFrame()
    rows: List[Dict[str, Any]] = []
    for _, hmm_row in selected_full.iterrows():
        hmm_id = str(hmm_row["hmm_id"])
        if hmm_id not in dataset_paths:
            continue
        try:
            sequences = load_dataset(dataset_paths[hmm_id])["train"][: cfg.N_TRAIN_MAIN]
            P = empirical_joint_matrix(
                sequences, h=cfg.EMPIRICAL_RANK_H, m=cfg.EMPIRICAL_RANK_M,
                vocab_size=cfg.VOCAB_SIZE, max_windows=cfg.EMPIRICAL_MAX_WINDOWS,
                seed=cfg.MASTER_SEED + stable_hash(hmm_id),
            )
            observed_all = singular_values_from_joint(P)
            # A centered rows-by-columns matrix has rank at most min(rows-1, cols-1).
            max_rank = max(0, min(P.shape[0] - 1, P.shape[1] - 1))
            observed = observed_all[:max_rank]
            null_values = []
            rng = np.random.default_rng(cfg.MASTER_SEED + stable_hash(hmm_id) + 77)
            n_eff = min(cfg.EMPIRICAL_MAX_WINDOWS,
                        sequences.shape[0] * (sequences.shape[1] - cfg.EMPIRICAL_RANK_H))
            p_null = np.outer(P.sum(axis=1), P.sum(axis=0)).reshape(-1)
            p_null = p_null / max(p_null.sum(), 1e-15)
            for _ in range(cfg.EMPIRICAL_NULL_REPEATS):
                sampled = rng.multinomial(max(int(n_eff), 1), p_null).reshape(P.shape)
                sampled = sampled / max(sampled.sum(), 1)
                null_values.append(singular_values_from_joint(sampled)[:max_rank])
            if max_rank == 0:
                threshold = np.array([], dtype=float)
                significant = np.array([], dtype=bool)
            else:
                null_pad = np.vstack([np.pad(x, (0, max_rank - len(x))) for x in null_values])
                # Bonferroni-adjusted per-component null threshold.
                q = 1.0 - 0.05 / max_rank
                threshold = np.quantile(null_pad, q, axis=0)
                numerical_floor = max(cfg.SPECTRAL_TOL,
                                      1e-8 * float(observed[0]) if len(observed) else cfg.SPECTRAL_TOL)
                significant = observed > np.maximum(threshold, numerical_floor)
            # Count only the leading contiguous significant singular directions.
            detected = 0
            for flag in significant:
                if not bool(flag):
                    break
                detected += 1
            true_rank_col = f"pred_h{cfg.EMPIRICAL_RANK_H}_m{cfg.EMPIRICAL_RANK_M}_r95"
            row: Dict[str, Any] = {
                "hmm_id": hmm_id, "K": int(hmm_row["K"]), "n_train": cfg.N_TRAIN_MAIN,
                "detected_rank": int(detected), "max_admissible_rank": int(max_rank),
                "true_r95": float(hmm_row.get(true_rank_col, np.nan)),
                "null_quantile": float(1.0 - 0.05 / max_rank) if max_rank else np.nan,
            }
            for i in range(max_rank):
                row[f"observed_sv{i+1}"] = float(observed[i]) if i < len(observed) else 0.0
                row[f"null_threshold_sv{i+1}"] = float(threshold[i]) if i < len(threshold) else 0.0
                row[f"significant_sv{i+1}"] = bool(significant[i]) if i < len(significant) else False
            rows.append(row)
        except Exception as exc:
            record_failure(paths, "empirical_rank", hmm_id, exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise
    out_df = pd.DataFrame(rows)
    atomic_write_csv(output, out_df)
    logger.info("Empirical-rank analysis completed for %d HMMs", len(out_df))
    return out_df


# -----------------------------------------------------------------------------
# Result aggregation and statistics
# -----------------------------------------------------------------------------

def collect_done_results(base: Path) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for path in base.rglob("DONE.json"):
        try:
            with path.open("r", encoding="utf-8") as handle:
                payload = json.load(handle)
            payload["result_file"] = str(path)
            rows.append(payload)
        except Exception:
            continue
    return pd.DataFrame(rows)


def attach_recovery_metrics(results: pd.DataFrame, metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    if results.empty:
        return results
    bayes_cols = ["hmm_id"] + [f"bayes_ce_h{h}" for h in range(1, cfg.MAX_CONTEXT + 1)]
    merged = results.merge(metrics[bayes_cols], on="hmm_id", how="left")
    H = cfg.MAX_CONTEXT
    shape_values = []
    excess_values = []
    recovery_values = []
    for _, row in merged.iterrows():
        model_shape = np.array(
            [float(row[f"test_ce_h{h}"]) - float(row[f"test_ce_h{H}"]) for h in range(1, H + 1)]
        )
        bayes_shape = np.array(
            [float(row[f"bayes_ce_h{h}"]) - float(row[f"bayes_ce_h{H}"]) for h in range(1, H + 1)]
        )
        shape_values.append(float(np.sqrt(np.mean((model_shape - bayes_shape) ** 2))))
        excess_values.append(float(row[f"test_ce_h{H}"] - row[f"bayes_ce_h{H}"]))
        bayes_gap = float(row["bayes_ce_h1"] - row[f"bayes_ce_h{H}"])
        model_gap = float(row["test_ce_h1"] - row[f"test_ce_h{H}"])
        recovery_values.append(model_gap / bayes_gap if abs(bayes_gap) > 1e-12 else np.nan)
    merged["shape_rmse"] = shape_values
    merged["excess_ce_H"] = excess_values
    merged["recovery_ratio"] = recovery_values
    return merged


def width_summary(run_metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    if run_metrics.empty:
        return pd.DataFrame()
    grouped = (
        run_metrics.groupby(["hmm_id", "architecture", "dimension"], as_index=False)
        .agg(
            shape_rmse=("shape_rmse", "mean"),
            shape_rmse_sd=("shape_rmse", "std"),
            excess_ce_H=("excess_ce_H", "mean"),
            recovery_ratio=("recovery_ratio", "mean"),
            best_val_ce=("best_val_ce", "mean"),
        )
    )
    rows: List[Dict[str, Any]] = []
    for (hmm_id, arch), group in grouped.groupby(["hmm_id", "architecture"]):
        group = group.sort_values("dimension")
        dims = group["dimension"].to_numpy(dtype=int)
        errors = group["shape_rmse"].to_numpy(dtype=float)
        cumulative_best = np.minimum.accumulate(errors)
        total_gain = float(cumulative_best[0] - cumulative_best[-1])
        def saturation(level: float) -> float:
            if total_gain <= 1e-8:
                return np.nan
            target = cumulative_best[0] - level * total_gain
            idx = int(np.argmax(cumulative_best <= target))
            return float(dims[idx])
        rows.append(
            {
                "hmm_id": hmm_id,
                "architecture": arch,
                "A_width_shape": float(np.mean(errors)),
                "best_shape_rmse": float(np.min(errors)),
                "best_dimension": int(dims[int(np.argmin(errors))]),
                "width_gain": total_gain,
                "d90": saturation(0.90),
                "d95": saturation(0.95),
                "mean_excess_ce_H": float(group["excess_ce_H"].mean()),
                "mean_recovery_ratio": float(group["recovery_ratio"].mean()),
            }
        )
    return pd.DataFrame(rows)


def cross_validated_model_comparison(
    summary: pd.DataFrame,
    metrics: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    if summary.empty:
        return pd.DataFrame()
    m = cfg.SELECTION_FUTURE_LENGTH
    rank_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_r95"
    sigma_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_sigma95"
    merged = summary.merge(metrics, on="hmm_id", how="left", suffixes=("", "_metric"))
    merged["log_K"] = np.log(np.clip(merged["K"].astype(float), 1e-12, None))
    merged["log_gain"] = np.log(np.clip(merged["context_gain_area"].astype(float), 1e-12, None))
    merged["log_rank"] = np.log(np.clip(merged[rank_col].astype(float), 1e-12, None))
    merged["neg_log_sigma"] = -np.log(np.clip(merged[sigma_col].astype(float), 1e-12, None))
    merged["log_tau"] = np.log(np.clip(merged["tau_observed_proxy"].astype(float), 1e-12, None))
    feature_sets = {
        "K_only": ["log_K"],
        "gain_only": ["log_gain"],
        "predictive_profile": ["log_gain", "log_rank", "neg_log_sigma", "log_tau"],
    }
    outcomes = ["A_width_shape", "best_shape_rmse", "mean_excess_ce_H"]
    rows: List[Dict[str, Any]] = []
    for arch, arch_df in merged.groupby("architecture"):
        for outcome in outcomes:
            data = arch_df.replace([np.inf, -np.inf], np.nan).dropna(
                subset=[outcome, "hmm_id", "K"] + list({x for cols in feature_sets.values() for x in cols})
            )
            if len(data) < 5:
                continue
            for model_name, features in feature_sets.items():
                for cv_name in ("leave_one_hmm_out", "leave_one_K_out"):
                    predictions = np.full(len(data), np.nan)
                    if cv_name == "leave_one_hmm_out":
                        folds = [(np.arange(len(data)) != i, np.array([i])) for i in range(len(data))]
                    else:
                        folds = []
                        for K in sorted(data["K"].unique()):
                            test_idx = np.where(data["K"].to_numpy() == K)[0]
                            train_mask = data["K"].to_numpy() != K
                            folds.append((train_mask, test_idx))
                    X = data[features].to_numpy(dtype=float)
                    y = data[outcome].to_numpy(dtype=float)
                    for train_mask, test_idx in folds:
                        train_idx = np.where(train_mask)[0]
                        if len(train_idx) <= len(features):
                            continue
                        model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
                        model.fit(X[train_idx], y[train_idx])
                        predictions[test_idx] = model.predict(X[test_idx])
                    valid = np.isfinite(predictions)
                    if valid.sum() < 3:
                        continue
                    rows.append(
                        {
                            "architecture": arch,
                            "outcome": outcome,
                            "model": model_name,
                            "cv": cv_name,
                            "n": int(valid.sum()),
                            "mae": float(mean_absolute_error(y[valid], predictions[valid])),
                            "rmse": float(np.sqrt(mean_squared_error(y[valid], predictions[valid]))),
                            "r2": float(r2_score(y[valid], predictions[valid])) if valid.sum() > 1 else np.nan,
                        }
                    )
    return pd.DataFrame(rows)


def bootstrap_mean_ci(values: np.ndarray, seed: int, repeats: int = 5000) -> Tuple[float, float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(repeats, len(values)), replace=True).mean(axis=1)
    return float(values.mean()), float(np.quantile(samples, 0.025)), float(np.quantile(samples, 0.975))


def matched_pair_effects(
    pairs: pd.DataFrame,
    width: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if pairs.empty or width.empty:
        return pd.DataFrame(), pd.DataFrame()
    lookup = width.set_index(["hmm_id", "architecture"]).to_dict("index")
    rows: List[Dict[str, Any]] = []
    for pair_index, pair in pairs.iterrows():
        for arch in cfg.ARCHITECTURES:
            key_a = (str(pair["hmm_a"]), arch)
            key_b = (str(pair["hmm_b"]), arch)
            if key_a not in lookup or key_b not in lookup:
                continue
            a, b = lookup[key_a], lookup[key_b]
            expected = str(pair.get("expected_harder", ""))
            if expected == str(pair["hmm_b"]):
                effect = float(b["A_width_shape"] - a["A_width_shape"])
            elif expected == str(pair["hmm_a"]):
                effect = float(a["A_width_shape"] - b["A_width_shape"])
            else:
                effect = abs(float(a["A_width_shape"] - b["A_width_shape"]))
            rows.append(
                {
                    "pair_index": int(pair_index),
                    "pair_type": pair["pair_type"],
                    "architecture": arch,
                    "hmm_a": pair["hmm_a"],
                    "hmm_b": pair["hmm_b"],
                    "A_width_a": float(a["A_width_shape"]),
                    "A_width_b": float(b["A_width_shape"]),
                    "signed_expected_effect": effect,
                }
            )
    effects = pd.DataFrame(rows)
    summaries: List[Dict[str, Any]] = []
    if not effects.empty:
        for (pair_type, arch), group in effects.groupby(["pair_type", "architecture"]):
            mean, low, high = bootstrap_mean_ci(
                group["signed_expected_effect"].to_numpy(),
                seed=cfg.MASTER_SEED + stable_hash(f"{pair_type}|{arch}"),
            )
            summaries.append(
                {
                    "pair_type": pair_type,
                    "architecture": arch,
                    "n_pairs": len(group),
                    "mean_effect": mean,
                    "ci_low": low,
                    "ci_high": high,
                }
            )
    return effects, pd.DataFrame(summaries)


def create_figures(
    run_metrics: pd.DataFrame,
    width: pd.DataFrame,
    metrics: pd.DataFrame,
    pairs: pd.DataFrame,
    sample_metrics: pd.DataFrame,
    empirical: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> None:
    if width.empty:
        return
    m = cfg.SELECTION_FUTURE_LENGTH
    rank_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_r95"
    sigma_col = f"pred_h{cfg.MAX_CONTEXT}_m{m}_sigma95"
    merged = width.merge(metrics, on="hmm_id", how="left")

    for arch, group in merged.groupby("architecture"):
        plt.figure(figsize=(7, 5))
        plt.scatter(group["K"], group["A_width_shape"])
        plt.xlabel("Hidden-state count K")
        plt.ylabel("Mean shape RMSE across widths")
        plt.title(f"K versus learner recovery difficulty ({arch})")
        plt.tight_layout()
        plt.savefig(paths["figures"] / f"K_vs_Awidth_{arch}.png", dpi=180)
        plt.close()

        plt.figure(figsize=(7, 5))
        x = (
            np.log(np.clip(group["context_gain_area"], 1e-12, None))
            + np.log(np.clip(group[rank_col], 1e-12, None))
            - np.log(np.clip(group[sigma_col], 1e-12, None))
            + np.log(np.clip(group["tau_observed_proxy"], 1e-12, None))
        )
        plt.scatter(x, group["A_width_shape"])
        plt.xlabel("Unfitted predictive-profile composite")
        plt.ylabel("Mean shape RMSE across widths")
        plt.title(f"Predictive profile versus recovery difficulty ({arch})")
        plt.tight_layout()
        plt.savefig(paths["figures"] / f"profile_vs_Awidth_{arch}.png", dpi=180)
        plt.close()

    # Width curves for the first few matched pairs.
    if not run_metrics.empty and not pairs.empty:
        grouped = (
            run_metrics.groupby(["hmm_id", "architecture", "dimension"], as_index=False)
            .agg(shape_rmse=("shape_rmse", "mean"))
        )
        for pair_index, pair in pairs.head(6).iterrows():
            for arch in cfg.ARCHITECTURES:
                subset = grouped[
                    (grouped["architecture"] == arch)
                    & grouped["hmm_id"].isin([pair["hmm_a"], pair["hmm_b"]])
                ]
                if subset.empty:
                    continue
                plt.figure(figsize=(7, 5))
                for hmm_id, curve in subset.groupby("hmm_id"):
                    curve = curve.sort_values("dimension")
                    plt.plot(curve["dimension"], curve["shape_rmse"], marker="o", label=hmm_id)
                plt.xscale("log", base=2)
                plt.xlabel("Hidden / embedding dimension")
                plt.ylabel("Centered shape RMSE")
                plt.title(f"{pair['pair_type']} pair {pair_index} ({arch})")
                plt.legend(fontsize=7)
                plt.tight_layout()
                plt.savefig(
                    paths["figures"] / f"pair_{pair_index}_{pair['pair_type']}_{arch}.png",
                    dpi=180,
                )
                plt.close()

    if not sample_metrics.empty:
        sample_grouped = (
            sample_metrics.groupby(["hmm_id", "architecture", "n_train"], as_index=False)
            .agg(excess_ce_H=("excess_ce_H", "mean"))
        )
        for arch, group in sample_grouped.groupby("architecture"):
            plt.figure(figsize=(8, 5))
            for hmm_id, curve in group.groupby("hmm_id"):
                curve = curve.sort_values("n_train")
                plt.plot(curve["n_train"], curve["excess_ce_H"], marker="o", label=hmm_id)
            plt.xscale("log", base=2)
            plt.xlabel("Number of training sequences")
            plt.ylabel("Excess CE at maximum context")
            plt.title(f"Sample-size sensitivity ({arch})")
            plt.legend(fontsize=6, ncol=2)
            plt.tight_layout()
            plt.savefig(paths["figures"] / f"sample_size_{arch}.png", dpi=180)
            plt.close()

    if not empirical.empty:
        plt.figure(figsize=(6, 5))
        plt.scatter(empirical["true_r95"], empirical["detected_rank"])
        low = min(empirical["true_r95"].min(), empirical["detected_rank"].min())
        high = max(empirical["true_r95"].max(), empirical["detected_rank"].max())
        plt.plot([low, high], [low, high], linestyle="--")
        plt.xlabel("True r95")
        plt.ylabel("Detected empirical rank")
        plt.title("True versus empirically detectable predictive rank")
        plt.tight_layout()
        plt.savefig(paths["figures"] / "true_vs_empirical_rank.png", dpi=180)
        plt.close()


def run_analysis(
    selected_full: pd.DataFrame,
    pairs_df: pd.DataFrame,
    empirical: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> None:
    if not cfg.RUN_ANALYSIS:
        return
    main_results = collect_done_results(paths["runs"])
    sample_results = collect_done_results(paths["sample_runs"])
    atomic_write_csv(paths["tables"] / "main_training_raw.csv", main_results)
    atomic_write_csv(paths["tables"] / "sample_size_training_raw.csv", sample_results)

    main_metrics = attach_recovery_metrics(main_results, selected_full, cfg)
    sample_metrics = attach_recovery_metrics(sample_results, selected_full, cfg)
    atomic_write_csv(paths["tables"] / "main_training_with_metrics.csv", main_metrics)
    atomic_write_csv(paths["tables"] / "sample_size_with_metrics.csv", sample_metrics)

    width = width_summary(main_metrics, cfg)
    atomic_write_csv(paths["tables"] / "width_summary.csv", width)
    cv = cross_validated_model_comparison(width, selected_full, cfg)
    atomic_write_csv(paths["tables"] / "model_comparison_cv.csv", cv)
    effects, effect_summary = matched_pair_effects(pairs_df, width, cfg)
    atomic_write_csv(paths["tables"] / "matched_pair_effects.csv", effects)
    atomic_write_csv(paths["tables"] / "matched_pair_effect_summary.csv", effect_summary)

    create_figures(
        main_metrics,
        width,
        selected_full,
        pairs_df,
        sample_metrics,
        empirical,
        cfg,
        paths,
    )
    logger.info("Analysis outputs written to %s", paths["analysis"])


# -----------------------------------------------------------------------------
# Main pipeline
# -----------------------------------------------------------------------------

def save_run_manifest(cfg: Config, paths: Dict[str, Path], device: torch.device) -> None:
    payload = {
        "config": asdict(cfg),
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    atomic_write_json(paths["config"] / "run_manifest.json", payload)


def main(cfg: Config = CFG) -> None:
    mount_drive_if_needed(cfg)
    configure_local_paths_for_non_colab(cfg)
    paths = project_paths(cfg)
    logger = setup_logger(paths["logs"])
    device = resolve_device(cfg)
    save_run_manifest(cfg, paths, device)
    set_global_seed(cfg.MASTER_SEED, cfg.DETERMINISTIC_TORCH)

    logger.info("Project root: %s", paths["root"])
    logger.info("Device: %s", device)
    logger.info("The pipeline is resumable. Completed conditions will be skipped automatically.")

    imported_df = import_existing_hmms(cfg, paths, logger)
    bank_df = generate_candidate_bank(cfg, paths, logger)
    frames = [frame for frame in (imported_df, bank_df) if not frame.empty]
    if not frames:
        raise RuntimeError("No HMMs are available. Enable bank generation or provide existing bundles.")
    all_metrics = pd.concat(frames, ignore_index=True, sort=False)
    all_metrics = all_metrics.drop_duplicates(subset=["hmm_id"], keep="last")
    atomic_write_csv(paths["bank"] / "all_hmm_metrics.csv", all_metrics)

    selected_df, pairs_df = select_matched_pairs(all_metrics, cfg, paths, logger)
    selected_full = materialize_selected_hmms(selected_df, cfg, paths, logger)
    if selected_full.empty:
        raise RuntimeError("No selected HMMs could be materialized")

    dataset_paths = prepare_all_datasets(selected_full, cfg, paths, logger)
    empirical = empirical_rank_analysis(selected_full, dataset_paths, cfg, paths, logger)

    run_training_grid(selected_full, dataset_paths, cfg, paths, logger, device)
    sample_hmms = choose_sample_size_hmms(pairs_df, selected_full, cfg)
    run_sample_size_grid(sample_hmms, dataset_paths, cfg, paths, logger, device)
    run_analysis(selected_full, pairs_df, empirical, cfg, paths, logger)

    completion = {
        "status": "pipeline_completed",
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "project_root": str(paths["root"]),
        "selected_hmms": int(len(selected_full)),
        "matched_pairs": int(len(pairs_df)),
        "main_done_conditions": int(len(collect_done_results(paths["runs"]))),
        "sample_done_conditions": int(len(collect_done_results(paths["sample_runs"]))),
    }
    atomic_write_json(paths["root"] / "PIPELINE_COMPLETE.json", completion)
    logger.info("Pipeline finished. Summary: %s", completion)




# =============================================================================
# Independent 64-HMM confirmatory cohort extension
# =============================================================================
#
# Purpose
# -------
# This extension creates a fully independent HMM cohort and evaluates, without
# tuning on the new learner outcomes, whether a pre-specified observable
# predictive profile predicts finite-sample neural recovery better than K alone.
#
# Design
# ------
# * K in {3,4,5,6}
# * 16 independently generated HMMs per K (64 HMMs total)
# * Selection uses only process-side features, never learner outcomes
# * GRU and causal Transformer
# * widths {8,32,128,256}
# * five seeds
# * 64 x 2 x 4 x 5 = 2560 neural conditions
# * fixed smoothed n-gram baseline
# * frozen holdout comparison learned on the earlier 24-HMM development cohort
# * pre-specified adjacent-K matching analysis on the new cohort
#
# The script is resumable. Re-running it skips completed neural conditions.
# All outputs are written to a new Google Drive folder and do not alter v2 or
# the strict matched-pair replication.

from itertools import product
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr


# -----------------------------------------------------------------------------
# User-editable settings
# -----------------------------------------------------------------------------

DEVELOPMENT_V2_ROOT = Path(
    "/content/drive/MyDrive/predictive_complexity_project/"
    "predictive_profile_experiment_v2"
)
DEVELOPMENT_STRICT_ROOT = Path(
    "/content/drive/MyDrive/predictive_complexity_project/"
    "predictive_profile_strict_replication_v1"
)
CONFIRM_ROOT = Path(
    "/content/drive/MyDrive/predictive_complexity_project/"
    "predictive_profile_confirmatory64_v1"
)

CONFIRM_MASTER_SEED = 2026071701
CONFIRM_K_VALUES = (3, 4, 5, 6)
CONFIRM_HMMS_PER_K = 16
CONFIRM_BANK_PER_K = 1000
CONFIRM_DIMENSIONS = (8, 32, 128, 256)
CONFIRM_SEEDS = (1, 2, 3, 4, 5)
CONFIRM_ARCHITECTURES = ("GRU", "Transformer")

# Run controls
PREPARE_ONLY = False
RUN_NGRAM_BASELINE = True
RUN_HOLDOUT_ANALYSIS = True
RUN_CROSS_K_CONFIRMATORY = True
FORCE_RESELECT_COHORT = False
FORCE_RETRAIN = False

# Optional chunking. None runs every remaining condition. Set, for example,
# 100 to run at most 100 new conditions before stopping cleanly.
MAX_NEW_CONDITIONS_PER_EXECUTION: Optional[int] = None

# Fixed confirmatory analysis settings. Do not change after looking at outcomes.
RIDGE_ALPHA = 1.0
PRIMARY_OUTCOMES = ("A_width_shape", "mean_excess_ce_H")
SECONDARY_OUTCOMES = ("best_shape_rmse",)
HOLDOUT_BOOTSTRAP_REPEATS = 10000
HOLDOUT_PERMUTATION_REPEATS = 10000

# Adjacent-K matching protocol
CROSS_K_ADJACENT_PAIRS = ((3, 4), (4, 5), (5, 6))
CROSS_K_TARGET_PAIRS = 12
CROSS_K_MAX_RMS_PROFILE_DISTANCE = 1.50
CROSS_K_RANDOM_REPEATS = 10000

# Fixed n-gram prior
NGRAM_ALPHA = 0.5

# Set this environment variable to 1 for a tiny local/Colab smoke test:
#   PP_CONFIRM_SMOKE_TEST=1
CONFIRM_SMOKE_TEST = os.environ.get("PP_CONFIRM_SMOKE_TEST", "0") == "1"


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

def make_confirm_config() -> Config:
    cfg = Config()
    cfg.PROJECT_ROOT = str(CONFIRM_ROOT)
    cfg.MASTER_SEED = CONFIRM_MASTER_SEED
    cfg.IMPORT_EXISTING_HMMS = False
    cfg.GENERATE_BANK = False  # independent bank is generated by this extension
    cfg.RUN_SAMPLE_SIZE_EXPERIMENT = False
    cfg.RUN_EMPIRICAL_RANK = False
    cfg.RUN_TRAINING = True
    cfg.RUN_ANALYSIS = True
    cfg.K_VALUES = CONFIRM_K_VALUES
    cfg.BANK_PER_K = CONFIRM_BANK_PER_K
    cfg.ARCHITECTURES = CONFIRM_ARCHITECTURES
    cfg.DIMENSIONS = CONFIRM_DIMENSIONS
    cfg.SEEDS = CONFIRM_SEEDS
    cfg.N_TRAIN_MAX = 256
    cfg.N_TRAIN_MAIN = 256
    cfg.N_VAL = 32
    cfg.N_TEST = 64
    cfg.SEQUENCE_LENGTH = 300
    cfg.EVAL_WINDOWS_PER_SEQUENCE = 8
    cfg.MAX_EPOCHS = 48
    cfg.MIN_EPOCHS = 12
    cfg.EARLY_STOP_PATIENCE = 8
    cfg.BATCH_SIZE = 16
    cfg.MATCH_PROTOCOL_VERSION = "independent_profile_kmeans_holdout_v1"
    cfg.CONTINUE_AFTER_ERROR = True
    cfg.RETRY_FAILED_CONDITIONS = True
    cfg.FORCE_RETRAIN = FORCE_RETRAIN

    if CONFIRM_SMOKE_TEST:
        cfg.K_VALUES = (3, 4)
        cfg.BANK_PER_K = 20
        cfg.ARCHITECTURES = ("GRU",)
        cfg.DIMENSIONS = (8,)
        cfg.SEEDS = (1,)
        cfg.N_TRAIN_MAX = 24
        cfg.N_TRAIN_MAIN = 16
        cfg.N_VAL = 8
        cfg.N_TEST = 8
        cfg.SEQUENCE_LENGTH = 40
        cfg.EVAL_WINDOWS_PER_SEQUENCE = 2
        cfg.MAX_EPOCHS = 2
        cfg.MIN_EPOCHS = 1
        cfg.EARLY_STOP_PATIENCE = 1
        cfg.BATCH_SIZE = 8
    return cfg


def effective_hmms_per_k(cfg: Config) -> int:
    return 4 if CONFIRM_SMOKE_TEST else CONFIRM_HMMS_PER_K


# -----------------------------------------------------------------------------
# Process-side marginal features and fixed feature transformations
# -----------------------------------------------------------------------------

def symbol_marginal_features_confirm(
    T: np.ndarray, O: np.ndarray, pi: np.ndarray
) -> Dict[str, Any]:
    p = np.asarray(pi @ O, dtype=np.float64)
    p = p / np.clip(p.sum(), 1e-15, None)
    entropy = float(-np.sum(p * np.log(np.clip(p, 1e-15, None))))
    uniform = np.full_like(p, 1.0 / len(p))
    output: Dict[str, Any] = {
        "symbol_entropy": entropy,
        "symbol_entropy_normalized": entropy / math.log(len(p)),
        "max_symbol_prob": float(p.max()),
        "min_symbol_prob_recomputed": float(p.min()),
        "marginal_l2_from_uniform": float(np.linalg.norm(p - uniform)),
    }
    for idx, value in enumerate(p):
        output[f"symbol_prob_{idx}"] = float(value)
    return output


def add_marginal_features_confirm(
    bank: pd.DataFrame,
    cfg: Config,
    cache_path: Path,
    logger: logging.Logger,
) -> pd.DataFrame:
    required = [f"symbol_prob_{i}" for i in range(cfg.VOCAB_SIZE)] + [
        "symbol_entropy",
        "min_symbol_prob_recomputed",
    ]
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if (
            len(cached) == len(bank)
            and set(cached["hmm_id"].astype(str)) == set(bank["hmm_id"].astype(str))
            and all(col in cached.columns for col in required)
        ):
            logger.info("Reusing marginal-feature cache: %s", cache_path)
            return cached

    rows: List[Dict[str, Any]] = []
    for count, (_, row) in enumerate(bank.iterrows(), start=1):
        T, O, pi = regenerate_from_row(row, cfg)
        rows.append({**row.to_dict(), **symbol_marginal_features_confirm(T, O, pi)})
        if count % 250 == 0:
            logger.info("Computed marginal features for %d/%d candidates", count, len(bank))
    out = pd.DataFrame(rows)
    atomic_write_csv(cache_path, out)
    return out


def add_selected_marginals_confirm(selected_full: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for _, row in selected_full.iterrows():
        path = Path(str(row["hmm_file"]))
        with np.load(path) as data:
            T, O, pi = data["T"], data["O"], data["pi"]
        rows.append({**row.to_dict(), **symbol_marginal_features_confirm(T, O, pi)})
    return pd.DataFrame(rows)


def profile_column_names(cfg: Config) -> Dict[str, str]:
    m = cfg.SELECTION_FUTURE_LENGTH
    return {
        "rank": f"pred_h{cfg.MAX_CONTEXT}_m{m}_r95",
        "sigma": f"pred_h{cfg.MAX_CONTEXT}_m{m}_sigma95",
        "signal": f"pred_h{cfg.MAX_CONTEXT}_m{m}_signal",
    }


def make_feature_frame(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    cols = profile_column_names(cfg)
    out = df.copy()
    out["log_K"] = np.log(np.clip(out["K"].astype(float), 1e-12, None))
    out["log_gain"] = np.log(
        np.clip(out["context_gain_area"].astype(float), 1e-12, None)
    )
    out["rank_value"] = out[cols["rank"]].astype(float)
    out["log_rank"] = np.log(np.clip(out[cols["rank"]].astype(float), 1e-12, None))
    out["neg_log_sigma"] = -np.log(
        np.clip(out[cols["sigma"]].astype(float), 1e-12, None)
    )
    out["log_tau"] = np.log(
        np.clip(out["tau_observed_proxy"].astype(float), 1e-12, None)
    )
    out["log_min_symbol_prob"] = np.log(
        np.clip(out["min_symbol_prob_recomputed"].astype(float), 1e-12, None)
    )
    out["log_signal"] = np.log(
        np.clip(out[cols["signal"]].astype(float), 1e-12, None)
    )
    return out


CORE_FEATURES = ("log_gain", "log_rank", "neg_log_sigma", "log_tau")
AUGMENTED_FEATURES = (
    "log_gain",
    "log_rank",
    "neg_log_sigma",
    "log_tau",
    "bayes_ce_h4",
    "symbol_entropy",
    "log_min_symbol_prob",
    "log_signal",
)
FEATURE_SETS: Dict[str, Tuple[str, ...]] = {
    "K_only": ("log_K",),
    "core_profile": CORE_FEATURES,
    "augmented_profile": AUGMENTED_FEATURES,
}


# -----------------------------------------------------------------------------
# Independent candidate bank
# -----------------------------------------------------------------------------

def old_generation_seeds() -> set[int]:
    path = DEVELOPMENT_V2_ROOT / "candidate_bank" / "candidate_bank_metrics.csv"
    if not path.exists():
        return set()
    try:
        old = pd.read_csv(path, usecols=["gen_seed"])
        return set(old["gen_seed"].dropna().astype(int).tolist())
    except Exception:
        return set()


def generate_independent_bank(
    cfg: Config, paths: Dict[str, Path], logger: logging.Logger
) -> pd.DataFrame:
    bank_csv = paths["bank"] / "candidate_bank_metrics.csv"
    if bank_csv.exists():
        bank_df = pd.read_csv(bank_csv)
    else:
        bank_df = pd.DataFrame()

    rows = bank_df.to_dict("records") if not bank_df.empty else []
    old_seeds = old_generation_seeds()

    for K in cfg.K_VALUES:
        existing_k = bank_df[bank_df.get("K", pd.Series(dtype=int)) == K].copy()
        accepted = len(existing_k)
        if accepted >= cfg.BANK_PER_K:
            logger.info("Independent bank K=%d already complete: %d", K, accepted)
            continue
        if not existing_k.empty and "bank_index" in existing_k.columns:
            index = int(existing_k["bank_index"].max()) + 1
        else:
            index = 0
        attempts = 0
        max_attempts = max(cfg.BANK_PER_K * 30, cfg.BANK_PER_K + 200)
        logger.info("Independent candidate bank K=%d: %d/%d", K, accepted, cfg.BANK_PER_K)

        while accepted < cfg.BANK_PER_K and attempts < max_attempts:
            params = generation_parameters(cfg, int(K), int(index))
            hmm_id = f"confirm_K{int(K)}_i{int(index):06d}"
            index += 1
            attempts += 1
            if int(params["gen_seed"]) in old_seeds:
                continue
            try:
                T, O, pi = generate_hmm(
                    K=int(K),
                    vocab_size=cfg.VOCAB_SIZE,
                    alpha_t=params["alpha_t"],
                    self_bias=params["self_bias"],
                    alpha_e=params["alpha_e"],
                    seed=params["gen_seed"],
                )
                valid, _ = validate_hmm(T, O, pi, cfg)
                if not valid:
                    continue
                metrics = compute_hmm_metrics(T, O, pi, cfg, cfg.BANK_FUTURE_LENGTHS)
                if float(metrics["context_gain_area"]) < cfg.MIN_CONTEXT_GAIN:
                    continue
                rows.append(
                    {
                        "hmm_id": hmm_id,
                        "source": "bank",
                        "cohort": "independent_confirmatory64",
                        **params,
                        **metrics,
                    }
                )
                accepted += 1
                if accepted % cfg.BANK_SAVE_EVERY == 0:
                    current = pd.DataFrame(rows).sort_values(["K", "hmm_id"])
                    atomic_write_csv(bank_csv, current)
                    logger.info("Saved independent bank checkpoint K=%d accepted=%d", K, accepted)
            except KeyboardInterrupt:
                current = pd.DataFrame(rows)
                if not current.empty:
                    atomic_write_csv(bank_csv, current.sort_values(["K", "hmm_id"]))
                raise
            except Exception as exc:
                record_failure(paths, "confirm_candidate_bank", hmm_id, exc)
                if not cfg.CONTINUE_AFTER_ERROR:
                    raise

        if accepted < cfg.BANK_PER_K:
            raise RuntimeError(
                f"Independent bank K={K} produced only {accepted}/{cfg.BANK_PER_K} valid HMMs"
            )
        current = pd.DataFrame(rows).sort_values(["K", "hmm_id"])
        atomic_write_csv(bank_csv, current)

    out = pd.DataFrame(rows).sort_values(["K", "hmm_id"]).reset_index(drop=True)
    logger.info("Independent candidate bank contains %d HMMs", len(out))
    return out


# -----------------------------------------------------------------------------
# Outcome-blind profile-space cohort selection
# -----------------------------------------------------------------------------

def selection_feature_matrix(group: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, np.ndarray]:
    transformed = make_feature_frame(group, cfg)
    selection_features = list(AUGMENTED_FEATURES)
    clean = transformed.replace([np.inf, -np.inf], np.nan).dropna(
        subset=selection_features
    ).copy()
    X = clean[selection_features].to_numpy(dtype=float)
    means = X.mean(axis=0)
    scales = X.std(axis=0, ddof=0)
    scales[scales < 1e-12] = 1.0
    Z = (X - means) / scales
    return clean, Z


def profile_coverage_table(
    bank_augmented: pd.DataFrame,
    selected: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    bank_f = make_feature_frame(bank_augmented, cfg)
    sel_f = make_feature_frame(selected, cfg)
    features = list(AUGMENTED_FEATURES)
    rows: List[Dict[str, Any]] = []
    for K in cfg.K_VALUES:
        for feature in features:
            for label, frame in (("bank", bank_f), ("selected", sel_f)):
                values = frame.loc[frame["K"] == K, feature].astype(float).to_numpy()
                values = values[np.isfinite(values)]
                if len(values) == 0:
                    continue
                rows.append(
                    {
                        "K": int(K),
                        "feature": feature,
                        "set": label,
                        "n": int(len(values)),
                        "min": float(np.min(values)),
                        "q10": float(np.quantile(values, 0.10)),
                        "median": float(np.median(values)),
                        "q90": float(np.quantile(values, 0.90)),
                        "max": float(np.max(values)),
                    }
                )
    return pd.DataFrame(rows)


def select_independent_cohort(
    bank_augmented: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> pd.DataFrame:
    selected_path = paths["selected"] / "confirmatory_selected_hmms.csv"
    lock_path = paths["selected"] / "SELECTION_LOCK.json"
    runs_exist = any(paths["runs"].rglob("DONE.json")) or any(paths["runs"].rglob("checkpoint.pt"))

    if selected_path.exists() and (not FORCE_RESELECT_COHORT or runs_exist):
        selected = pd.read_csv(selected_path)
        logger.info("Reusing locked independent cohort selection: %s", selected_path)
        return selected
    if FORCE_RESELECT_COHORT and runs_exist:
        raise RuntimeError("Refusing to reselect the cohort after neural training has started")

    n_per_k = effective_hmms_per_k(cfg)
    rows: List[pd.DataFrame] = []
    for K in cfg.K_VALUES:
        group = bank_augmented[bank_augmented["K"] == K].copy().reset_index(drop=True)
        clean, Z = selection_feature_matrix(group, cfg)
        if len(clean) < n_per_k:
            raise RuntimeError(f"K={K}: only {len(clean)} finite candidates for {n_per_k} clusters")

        kmeans = KMeans(
            n_clusters=n_per_k,
            random_state=cfg.MASTER_SEED + int(K),
            n_init=20,
            max_iter=500,
        )
        labels = kmeans.fit_predict(Z)
        selected_indices: List[int] = []
        diagnostics: Dict[int, Tuple[int, float, int]] = {}
        for cluster_id in range(n_per_k):
            idx = np.where(labels == cluster_id)[0]
            center = kmeans.cluster_centers_[cluster_id]
            distances = np.sqrt(np.mean((Z[idx] - center) ** 2, axis=1))
            local = int(idx[int(np.argmin(distances))])
            selected_indices.append(local)
            diagnostics[local] = (
                int(cluster_id),
                float(np.min(distances)),
                int(len(idx)),
            )

        chosen = clean.iloc[selected_indices].copy()
        chosen["selection_cluster"] = [diagnostics[i][0] for i in selected_indices]
        chosen["distance_to_cluster_center"] = [diagnostics[i][1] for i in selected_indices]
        chosen["cluster_size"] = [diagnostics[i][2] for i in selected_indices]
        chosen["selection_protocol"] = cfg.MATCH_PROTOCOL_VERSION
        rows.append(chosen)

    selected = pd.concat(rows, ignore_index=True)
    selected = selected.sort_values(["K", "selection_cluster"]).reset_index(drop=True)
    expected = len(cfg.K_VALUES) * n_per_k
    if len(selected) != expected or selected["hmm_id"].nunique() != expected:
        raise AssertionError(f"Expected {expected} unique selected HMMs; got {len(selected)}")

    atomic_write_csv(selected_path, selected)
    coverage = profile_coverage_table(bank_augmented, selected, cfg)
    atomic_write_csv(paths["tables"] / "profile_coverage_summary.csv", coverage)

    selection_hash = hashlib.sha256(
        "|".join(selected["hmm_id"].astype(str).tolist()).encode("utf-8")
    ).hexdigest()
    atomic_write_json(
        lock_path,
        {
            "status": "locked_before_training",
            "selection_protocol": cfg.MATCH_PROTOCOL_VERSION,
            "master_seed": cfg.MASTER_SEED,
            "selected_hmms": int(len(selected)),
            "hmm_ids_sha256": selection_hash,
            "hmm_ids": selected["hmm_id"].astype(str).tolist(),
            "locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
    )
    logger.info("Selected and locked %d independent HMMs", len(selected))
    return selected


# -----------------------------------------------------------------------------
# Chunkable neural training
# -----------------------------------------------------------------------------

def run_confirmatory_training_grid(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
    device: torch.device,
) -> Tuple[int, int]:
    conditions = [
        (str(row["hmm_id"]), arch, int(dim), int(seed))
        for _, row in selected_full.iterrows()
        for arch in cfg.ARCHITECTURES
        for dim in cfg.DIMENSIONS
        for seed in cfg.SEEDS
    ]
    expected = len(conditions)
    completed_before = len(collect_done_results(paths["runs"]))
    remaining = []
    for hmm_id, arch, dim, seed in conditions:
        run_dir = condition_directory(
            paths["runs"], hmm_id, arch, dim, seed, cfg.N_TRAIN_MAIN
        )
        if not (run_dir / "DONE.json").exists() or cfg.FORCE_RETRAIN:
            remaining.append((hmm_id, arch, dim, seed))

    if MAX_NEW_CONDITIONS_PER_EXECUTION is not None:
        remaining = remaining[: int(MAX_NEW_CONDITIONS_PER_EXECUTION)]
    logger.info(
        "Confirmatory grid expected=%d completed_before=%d scheduled_now=%d",
        expected,
        completed_before,
        len(remaining),
    )

    for idx, (hmm_id, arch, dim, seed) in enumerate(remaining, start=1):
        logger.info(
            "Confirmatory condition %d/%d: %s %s d=%d seed=%d",
            idx,
            len(remaining),
            hmm_id,
            arch,
            dim,
            seed,
        )
        try:
            train_one_condition(
                hmm_id=hmm_id,
                architecture=arch,
                dimension=dim,
                seed=seed,
                n_train=cfg.N_TRAIN_MAIN,
                dataset_path=dataset_paths[hmm_id],
                run_base=paths["runs"],
                cfg=cfg,
                device=device,
                logger=logger,
            )
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            record_failure(paths, "confirm_training", f"{hmm_id}_{arch}_d{dim}_s{seed}", exc)
            logger.error("Condition failed: %s", exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise

    completed_after = len(collect_done_results(paths["runs"]))
    return completed_after, expected


# -----------------------------------------------------------------------------
# Smoothed n-gram baseline
# -----------------------------------------------------------------------------

def fit_ngram_counts(
    train_sequences: np.ndarray,
    h: int,
    vocab_size: int,
) -> Dict[Tuple[int, ...], np.ndarray]:
    counts: Dict[Tuple[int, ...], np.ndarray] = {}
    for seq in train_sequences:
        seq = np.asarray(seq, dtype=np.int64)
        for end in range(h, len(seq)):
            context = tuple(int(x) for x in seq[end - h : end])
            if context not in counts:
                counts[context] = np.zeros(vocab_size, dtype=np.int64)
            counts[context][int(seq[end])] += 1
    return counts


def evaluate_ngram_ce(
    counts: Dict[Tuple[int, ...], np.ndarray],
    test_sequences: np.ndarray,
    h: int,
    cfg: Config,
    seed: int,
    alpha: float,
) -> float:
    x, _, y = fixed_eval_windows(
        test_sequences,
        h=h,
        max_context=cfg.MAX_CONTEXT,
        vocab_size=cfg.VOCAB_SIZE,
        windows_per_sequence=cfg.EVAL_WINDOWS_PER_SEQUENCE,
        seed=seed,
    )
    total = 0.0
    for row, target in zip(x, y):
        context = tuple(int(v) for v in row[:h])
        c = counts.get(context)
        if c is None:
            denominator = cfg.VOCAB_SIZE * alpha
            probability = alpha / denominator
        else:
            denominator = float(c.sum()) + cfg.VOCAB_SIZE * alpha
            probability = (float(c[int(target)]) + alpha) / denominator
        total -= math.log(max(probability, 1e-15))
    return total / max(len(y), 1)


def run_ngram_baseline(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> pd.DataFrame:
    out_path = paths["tables"] / "ngram_baseline.csv"
    if out_path.exists():
        existing = pd.read_csv(out_path)
        if len(existing) == len(selected_full):
            logger.info("Reusing n-gram baseline: %s", out_path)
            return existing

    rows: List[Dict[str, Any]] = []
    metric_lookup = selected_full.set_index("hmm_id", drop=False).to_dict("index")
    for index, (hmm_id, dataset_path) in enumerate(dataset_paths.items(), start=1):
        data = load_dataset(dataset_path)
        train = data["train"][: cfg.N_TRAIN_MAIN]
        test = data["test"][: cfg.N_TEST]
        curve: Dict[int, float] = {}
        for h in range(1, cfg.MAX_CONTEXT + 1):
            counts = fit_ngram_counts(train, h, cfg.VOCAB_SIZE)
            curve[h] = evaluate_ngram_ce(
                counts,
                test,
                h,
                cfg,
                seed=cfg.MASTER_SEED + stable_hash(f"ngram|{hmm_id}|h{h}"),
                alpha=NGRAM_ALPHA,
            )
        metric = metric_lookup[hmm_id]
        H = cfg.MAX_CONTEXT
        model_shape = np.array([curve[h] - curve[H] for h in range(1, H + 1)])
        bayes_shape = np.array(
            [float(metric[f"bayes_ce_h{h}"]) - float(metric[f"bayes_ce_h{H}"]) for h in range(1, H + 1)]
        )
        bayes_gap = float(metric["bayes_ce_h1"] - metric[f"bayes_ce_h{H}"])
        model_gap = float(curve[1] - curve[H])
        row: Dict[str, Any] = {
            "hmm_id": hmm_id,
            "K": int(metric["K"]),
            "alpha": float(NGRAM_ALPHA),
            "shape_rmse": float(np.sqrt(np.mean((model_shape - bayes_shape) ** 2))),
            "excess_ce_H": float(curve[H] - metric[f"bayes_ce_h{H}"]),
            "recovery_ratio": model_gap / bayes_gap if abs(bayes_gap) > 1e-12 else np.nan,
        }
        for h, value in curve.items():
            row[f"test_ce_h{h}"] = float(value)
        rows.append(row)
        if index % 8 == 0:
            logger.info("Computed n-gram baseline for %d/%d HMMs", index, len(dataset_paths))

    out = pd.DataFrame(rows)
    atomic_write_csv(out_path, out)
    return out


# -----------------------------------------------------------------------------
# Frozen development-to-confirmatory holdout comparison
# -----------------------------------------------------------------------------

def locate_development_augmented_metrics() -> pd.DataFrame:
    candidates = [
        DEVELOPMENT_STRICT_ROOT
        / "analysis"
        / "existing_v2_additional"
        / "existing_selected_metrics_augmented.csv",
        DEVELOPMENT_V2_ROOT / "selected_hmms" / "selected_hmm_metrics_full.csv",
    ]
    for path in candidates:
        if not path.exists():
            continue
        df = pd.read_csv(path)
        required = {"symbol_entropy", "min_symbol_prob_recomputed"}
        if required.issubset(df.columns):
            return df
        if "hmm_file" in df.columns:
            return add_selected_marginals_confirm(df)
    raise FileNotFoundError(
        "Could not locate augmented metrics for the earlier 24-HMM development cohort"
    )


def build_matching_width_summary_from_v2(cfg: Config) -> pd.DataFrame:
    path = DEVELOPMENT_V2_ROOT / "analysis" / "tables" / "main_training_with_metrics.csv"
    if not path.exists():
        raise FileNotFoundError(f"Development run-level metrics not found: {path}")
    run_metrics = pd.read_csv(path)
    run_metrics = run_metrics[
        run_metrics["dimension"].astype(int).isin([int(x) for x in cfg.DIMENSIONS])
    ].copy()
    expected_dims = set(int(x) for x in cfg.DIMENSIONS)
    observed = set(run_metrics["dimension"].dropna().astype(int).unique().tolist())
    if not expected_dims.issubset(observed):
        raise RuntimeError(
            f"Development data lacks matching widths: expected {sorted(expected_dims)}, observed {sorted(observed)}"
        )
    return width_summary(run_metrics, cfg)


def merged_outcome_feature_table(
    summary: pd.DataFrame,
    metrics: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    merged = summary.merge(metrics, on="hmm_id", how="left", suffixes=("", "_metric"))
    return make_feature_frame(merged, cfg)


def fit_frozen_models_and_predict(
    development: pd.DataFrame,
    confirmatory: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    predictions: List[Dict[str, Any]] = []
    coefficients: List[Dict[str, Any]] = []
    performance: List[Dict[str, Any]] = []
    outcomes = tuple(PRIMARY_OUTCOMES) + tuple(SECONDARY_OUTCOMES)

    for architecture in cfg.ARCHITECTURES:
        dev_arch = development[development["architecture"] == architecture].copy()
        test_arch = confirmatory[confirmatory["architecture"] == architecture].copy()
        for outcome in outcomes:
            for model_name, features in FEATURE_SETS.items():
                dev = dev_arch.replace([np.inf, -np.inf], np.nan).dropna(
                    subset=[outcome] + list(features)
                )
                test = test_arch.replace([np.inf, -np.inf], np.nan).dropna(
                    subset=[outcome, "hmm_id", "K"] + list(features)
                )
                if len(dev) <= len(features) or len(test) == 0:
                    continue
                pipeline = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
                pipeline.fit(dev[list(features)].to_numpy(float), dev[outcome].to_numpy(float))
                pred = pipeline.predict(test[list(features)].to_numpy(float))
                y = test[outcome].to_numpy(float)

                for (_, row), p in zip(test.iterrows(), pred):
                    predictions.append(
                        {
                            "hmm_id": row["hmm_id"],
                            "K": int(row["K"]),
                            "architecture": architecture,
                            "outcome": outcome,
                            "model": model_name,
                            "observed": float(row[outcome]),
                            "predicted": float(p),
                            "error": float(p - row[outcome]),
                            "absolute_error": float(abs(p - row[outcome])),
                            "squared_error": float((p - row[outcome]) ** 2),
                        }
                    )

                performance.append(
                    {
                        "architecture": architecture,
                        "outcome": outcome,
                        "model": model_name,
                        "n_development": int(len(dev)),
                        "n_confirmatory": int(len(test)),
                        "mae": float(mean_absolute_error(y, pred)),
                        "rmse": float(np.sqrt(mean_squared_error(y, pred))),
                        "r2": float(r2_score(y, pred)),
                    }
                )
                scaler = pipeline.named_steps["standardscaler"]
                ridge = pipeline.named_steps["ridge"]
                for feature, mean, scale, coef in zip(
                    features, scaler.mean_, scaler.scale_, ridge.coef_
                ):
                    coefficients.append(
                        {
                            "architecture": architecture,
                            "outcome": outcome,
                            "model": model_name,
                            "feature": feature,
                            "development_mean": float(mean),
                            "development_scale": float(scale),
                            "standardized_coefficient": float(coef),
                            "ridge_alpha": float(RIDGE_ALPHA),
                            "intercept": float(ridge.intercept_),
                        }
                    )

    pred_df = pd.DataFrame(predictions)
    perf_df = pd.DataFrame(performance)
    coef_df = pd.DataFrame(coefficients)
    atomic_write_csv(paths["tables"] / "confirmatory_holdout_predictions.csv", pred_df)
    atomic_write_csv(paths["tables"] / "confirmatory_holdout_performance.csv", perf_df)
    atomic_write_csv(paths["tables"] / "frozen_development_model_coefficients.csv", coef_df)
    return pred_df, perf_df, coef_df


def paired_holdout_model_comparisons(
    predictions: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    rng = np.random.default_rng(cfg.MASTER_SEED + 707)
    for architecture in cfg.ARCHITECTURES:
        for outcome in PRIMARY_OUTCOMES:
            sub = predictions[
                (predictions["architecture"] == architecture)
                & (predictions["outcome"] == outcome)
            ]
            wide = sub.pivot(index="hmm_id", columns="model", values="squared_error")
            if "K_only" not in wide.columns:
                continue
            for model_name in ("core_profile", "augmented_profile"):
                if model_name not in wide.columns:
                    continue
                pair = wide[["K_only", model_name]].dropna()
                improvement = pair["K_only"].to_numpy() - pair[model_name].to_numpy()
                if len(improvement) == 0:
                    continue
                boot = rng.choice(
                    improvement,
                    size=(HOLDOUT_BOOTSTRAP_REPEATS, len(improvement)),
                    replace=True,
                ).mean(axis=1)
                signs = rng.choice(
                    np.array([-1.0, 1.0]),
                    size=(HOLDOUT_PERMUTATION_REPEATS, len(improvement)),
                    replace=True,
                )
                perm = (signs * improvement[None, :]).mean(axis=1)
                observed = float(improvement.mean())
                p_two = float((np.sum(np.abs(perm) >= abs(observed)) + 1) / (len(perm) + 1))
                rows.append(
                    {
                        "architecture": architecture,
                        "outcome": outcome,
                        "comparison": f"{model_name}_vs_K_only",
                        "n_hmms": int(len(improvement)),
                        "mean_squared_error_improvement": observed,
                        "median_squared_error_improvement": float(np.median(improvement)),
                        "bootstrap_ci_low": float(np.quantile(boot, 0.025)),
                        "bootstrap_ci_high": float(np.quantile(boot, 0.975)),
                        "sign_flip_p_two_sided": p_two,
                        "fraction_hmms_improved": float(np.mean(improvement > 0)),
                    }
                )
    out = pd.DataFrame(rows)
    atomic_write_csv(paths["tables"] / "confirmatory_holdout_paired_comparisons.csv", out)
    return out


# -----------------------------------------------------------------------------
# Adjacent-K confirmatory matching
# -----------------------------------------------------------------------------

def development_augmented_scaler(
    development_metrics: pd.DataFrame,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray]:
    dev = make_feature_frame(development_metrics, cfg).replace([np.inf, -np.inf], np.nan)
    dev = dev.dropna(subset=list(AUGMENTED_FEATURES))
    X = dev[list(AUGMENTED_FEATURES)].to_numpy(float)
    mean = X.mean(axis=0)
    scale = X.std(axis=0, ddof=0)
    scale[scale < 1e-12] = 1.0
    return mean, scale


def make_cross_k_pairs(
    selected_augmented: pd.DataFrame,
    development_metrics: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> pd.DataFrame:
    transformed = make_feature_frame(selected_augmented, cfg)
    mean, scale = development_augmented_scaler(development_metrics, cfg)
    rows: List[Dict[str, Any]] = []

    for K_a, K_b in CROSS_K_ADJACENT_PAIRS:
        a = transformed[transformed["K"] == K_a].copy().reset_index(drop=True)
        b = transformed[transformed["K"] == K_b].copy().reset_index(drop=True)
        if a.empty or b.empty:
            continue
        X_a = (a[list(AUGMENTED_FEATURES)].to_numpy(float) - mean) / scale
        X_b = (b[list(AUGMENTED_FEATURES)].to_numpy(float) - mean) / scale
        distances = np.sqrt(
            np.mean((X_a[:, None, :] - X_b[None, :, :]) ** 2, axis=2)
        )
        ia, ib = linear_sum_assignment(distances)
        assigned = sorted(
            [(int(i), int(j), float(distances[i, j])) for i, j in zip(ia, ib)],
            key=lambda x: x[2],
        )
        accepted = [x for x in assigned if x[2] <= CROSS_K_MAX_RMS_PROFILE_DISTANCE]
        accepted = accepted[: min(CROSS_K_TARGET_PAIRS, len(accepted))]
        for pair_index, (i, j, distance) in enumerate(accepted):
            rows.append(
                {
                    "K_a": int(K_a),
                    "K_b": int(K_b),
                    "pair_index": int(pair_index),
                    "hmm_a": str(a.iloc[i]["hmm_id"]),
                    "hmm_b": str(b.iloc[j]["hmm_id"]),
                    "profile_rms_distance": float(distance),
                    "matching_features": "|".join(AUGMENTED_FEATURES),
                    "caliper": float(CROSS_K_MAX_RMS_PROFILE_DISTANCE),
                    "target_pairs": int(CROSS_K_TARGET_PAIRS),
                }
            )
    out = pd.DataFrame(rows)
    atomic_write_csv(paths["tables"] / "crossK_confirmatory_pairs.csv", out)
    return out


def cross_k_confirmatory_test(
    pairs: pd.DataFrame,
    width: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if pairs.empty:
        empty = pd.DataFrame()
        atomic_write_csv(paths["tables"] / "crossK_confirmatory_details.csv", empty)
        atomic_write_csv(paths["tables"] / "crossK_confirmatory_summary.csv", empty)
        return empty, empty

    rng = np.random.default_rng(cfg.MASTER_SEED + 909)
    details: List[Dict[str, Any]] = []
    summaries: List[Dict[str, Any]] = []
    outcomes = tuple(PRIMARY_OUTCOMES) + tuple(SECONDARY_OUTCOMES)

    for architecture in cfg.ARCHITECTURES:
        arch = width[width["architecture"] == architecture].set_index("hmm_id")
        for outcome in outcomes:
            for (K_a, K_b), pair_group in pairs.groupby(["K_a", "K_b"]):
                pair_group = pair_group.copy()
                matched_values: List[float] = []
                for _, pair in pair_group.iterrows():
                    a_id, b_id = str(pair["hmm_a"]), str(pair["hmm_b"])
                    if a_id not in arch.index or b_id not in arch.index:
                        continue
                    value = abs(float(arch.loc[a_id, outcome]) - float(arch.loc[b_id, outcome]))
                    matched_values.append(value)
                    details.append(
                        {
                            "architecture": architecture,
                            "outcome": outcome,
                            "K_a": int(K_a),
                            "K_b": int(K_b),
                            "hmm_a": a_id,
                            "hmm_b": b_id,
                            "profile_rms_distance": float(pair["profile_rms_distance"]),
                            "outcome_absolute_difference": float(value),
                        }
                    )
                n_pairs = len(matched_values)
                if n_pairs == 0:
                    continue
                available_a = [x for x in arch.index if int(arch.loc[x, "K"]) == int(K_a)] if "K" in arch.columns else []
                available_b = [x for x in arch.index if int(arch.loc[x, "K"]) == int(K_b)] if "K" in arch.columns else []
                # width_summary does not always retain K; recover sets from matched IDs when needed.
                if not available_a or not available_b:
                    selected_a = sorted(set(pair_group["hmm_a"].astype(str)))
                    selected_b = sorted(set(pair_group["hmm_b"].astype(str)))
                    # Random baseline uses all cohort IDs from the corresponding K, supplied below
                    available_a = selected_a
                    available_b = selected_b

                random_means = np.empty(CROSS_K_RANDOM_REPEATS, dtype=float)
                for r in range(CROSS_K_RANDOM_REPEATS):
                    sample_a = rng.choice(available_a, size=n_pairs, replace=len(available_a) < n_pairs)
                    sample_b = rng.choice(available_b, size=n_pairs, replace=len(available_b) < n_pairs)
                    random_means[r] = np.mean(
                        [abs(float(arch.loc[a_id, outcome]) - float(arch.loc[b_id, outcome])) for a_id, b_id in zip(sample_a, sample_b)]
                    )
                observed = float(np.mean(matched_values))
                p_lower = float((np.sum(random_means <= observed) + 1) / (len(random_means) + 1))
                summaries.append(
                    {
                        "architecture": architecture,
                        "outcome": outcome,
                        "K_a": int(K_a),
                        "K_b": int(K_b),
                        "n_pairs": int(n_pairs),
                        "matched_mean_difference": observed,
                        "random_mean_difference": float(np.mean(random_means)),
                        "matched_to_random_ratio": observed / max(float(np.mean(random_means)), 1e-15),
                        "random_ci_low": float(np.quantile(random_means, 0.025)),
                        "random_ci_high": float(np.quantile(random_means, 0.975)),
                        "permutation_p_lower_is_better": p_lower,
                    }
                )

    details_df = pd.DataFrame(details)
    summary_df = pd.DataFrame(summaries)
    atomic_write_csv(paths["tables"] / "crossK_confirmatory_details.csv", details_df)
    atomic_write_csv(paths["tables"] / "crossK_confirmatory_summary.csv", summary_df)
    return details_df, summary_df


# -----------------------------------------------------------------------------
# Figures and compact synthesis
# -----------------------------------------------------------------------------

def create_confirmatory_figures(
    predictions: pd.DataFrame,
    performance: pd.DataFrame,
    width: pd.DataFrame,
    selected_augmented: pd.DataFrame,
    ngram: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> None:
    if not predictions.empty:
        for architecture in cfg.ARCHITECTURES:
            for outcome in PRIMARY_OUTCOMES:
                sub = predictions[
                    (predictions["architecture"] == architecture)
                    & (predictions["outcome"] == outcome)
                ]
                if sub.empty:
                    continue
                plt.figure(figsize=(6.5, 5.5))
                for model_name, group in sub.groupby("model"):
                    plt.scatter(group["observed"], group["predicted"], alpha=0.75, label=model_name)
                low = float(min(sub["observed"].min(), sub["predicted"].min()))
                high = float(max(sub["observed"].max(), sub["predicted"].max()))
                plt.plot([low, high], [low, high], linestyle="--")
                plt.xlabel("Observed confirmatory outcome")
                plt.ylabel("Frozen-model prediction")
                plt.title(f"{architecture}: {outcome}")
                plt.legend()
                plt.tight_layout()
                plt.savefig(
                    paths["figures"] / f"holdout_prediction_{architecture}_{outcome}.png",
                    dpi=220,
                )
                plt.close()

    if not performance.empty:
        for architecture in cfg.ARCHITECTURES:
            for outcome in PRIMARY_OUTCOMES:
                sub = performance[
                    (performance["architecture"] == architecture)
                    & (performance["outcome"] == outcome)
                ].copy()
                if sub.empty:
                    continue
                plt.figure(figsize=(6.5, 4.5))
                positions = np.arange(len(sub))
                plt.bar(positions, sub["rmse"])
                plt.xticks(positions, sub["model"], rotation=20)
                plt.ylabel("Holdout RMSE")
                plt.title(f"{architecture}: {outcome}")
                plt.tight_layout()
                plt.savefig(
                    paths["figures"] / f"holdout_rmse_{architecture}_{outcome}.png",
                    dpi=220,
                )
                plt.close()

    if not width.empty and not ngram.empty:
        neural = width.groupby("hmm_id", as_index=False)["A_width_shape"].mean()
        merged = neural.merge(ngram[["hmm_id", "shape_rmse"]], on="hmm_id", how="inner")
        if len(merged) >= 3:
            plt.figure(figsize=(6.0, 5.0))
            plt.scatter(merged["shape_rmse"], merged["A_width_shape"], alpha=0.8)
            plt.xlabel("Smoothed n-gram Shape RMSE")
            plt.ylabel("Mean neural A_width_shape")
            rho, _ = spearmanr(merged["shape_rmse"], merged["A_width_shape"])
            plt.title(f"Count-based vs neural difficulty (Spearman={rho:.3f})")
            plt.tight_layout()
            plt.savefig(paths["figures"] / "ngram_vs_neural_difficulty.png", dpi=220)
            plt.close()


def ngram_neural_association(
    width: pd.DataFrame,
    ngram: pd.DataFrame,
    paths: Dict[str, Path],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    if ngram.empty:
        out = pd.DataFrame()
        atomic_write_csv(paths["tables"] / "ngram_neural_association.csv", out)
        return out
    for architecture, group in width.groupby("architecture"):
        merged = group.merge(ngram, on="hmm_id", suffixes=("_neural", "_ngram"))
        for neural_col, ngram_col in (
            ("A_width_shape", "shape_rmse"),
            ("mean_excess_ce_H", "excess_ce_H"),
        ):
            valid = merged[[neural_col, ngram_col]].dropna()
            if len(valid) < 3:
                continue
            rho, p = spearmanr(valid[neural_col], valid[ngram_col])
            rows.append(
                {
                    "architecture": architecture,
                    "neural_outcome": neural_col,
                    "ngram_outcome": ngram_col,
                    "n": int(len(valid)),
                    "spearman_rho": float(rho),
                    "p_value": float(p),
                }
            )
    out = pd.DataFrame(rows)
    atomic_write_csv(paths["tables"] / "ngram_neural_association.csv", out)
    return out


# -----------------------------------------------------------------------------
# Protocol and completion records
# -----------------------------------------------------------------------------

def write_confirmatory_protocol(cfg: Config, paths: Dict[str, Path]) -> None:
    protocol = {
        "study": "independent_64_HMM_confirmatory_cohort",
        "locked_before_learner_outcomes": True,
        "master_seed": cfg.MASTER_SEED,
        "K_values": list(cfg.K_VALUES),
        "HMMs_per_K": effective_hmms_per_k(cfg),
        "selection": {
            "method": "KMeans coverage in standardized augmented profile space within each K",
            "features": list(AUGMENTED_FEATURES),
            "learner_outcomes_used": False,
        },
        "neural_grid": {
            "architectures": list(cfg.ARCHITECTURES),
            "dimensions": list(cfg.DIMENSIONS),
            "seeds": list(cfg.SEEDS),
            "n_train": cfg.N_TRAIN_MAIN,
            "n_val": cfg.N_VAL,
            "n_test": cfg.N_TEST,
            "sequence_length": cfg.SEQUENCE_LENGTH,
        },
        "frozen_models": {
            "development_cohort": "earlier 24-HMM v2 cohort",
            "ridge_alpha": RIDGE_ALPHA,
            "feature_sets": {k: list(v) for k, v in FEATURE_SETS.items()},
            "primary_outcomes": list(PRIMARY_OUTCOMES),
            "secondary_outcomes": list(SECONDARY_OUTCOMES),
            "no_refitting_on_confirmatory_outcomes": True,
        },
        "cross_K": {
            "K_pairs": [list(x) for x in CROSS_K_ADJACENT_PAIRS],
            "target_pairs": CROSS_K_TARGET_PAIRS,
            "profile_rms_caliper": CROSS_K_MAX_RMS_PROFILE_DISTANCE,
            "matching_features": list(AUGMENTED_FEATURES),
        },
        "ngram_alpha": NGRAM_ALPHA,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    path = paths["config"] / "CONFIRMATORY_PROTOCOL.json"
    if not path.exists():
        atomic_write_json(path, protocol)


# -----------------------------------------------------------------------------
# Main entry point
# -----------------------------------------------------------------------------

def confirmatory_main() -> None:
    cfg = make_confirm_config()
    mount_drive_if_needed(cfg)
    configure_local_paths_for_non_colab(cfg)
    paths = project_paths(cfg)
    logger = setup_logger(paths["logs"])
    device = resolve_device(cfg)
    set_global_seed(cfg.MASTER_SEED, cfg.DETERMINISTIC_TORCH)
    save_run_manifest(cfg, paths, device)
    write_confirmatory_protocol(cfg, paths)

    n_per_k = effective_hmms_per_k(cfg)
    expected_hmms = len(cfg.K_VALUES) * n_per_k
    expected_runs = expected_hmms * len(cfg.ARCHITECTURES) * len(cfg.DIMENSIONS) * len(cfg.SEEDS)
    logger.info("Confirmatory root: %s", paths["root"])
    logger.info("Device: %s", device)
    logger.info(
        "Requested design: %d HMMs x %d architectures x %d widths x %d seeds = %d conditions",
        expected_hmms,
        len(cfg.ARCHITECTURES),
        len(cfg.DIMENSIONS),
        len(cfg.SEEDS),
        expected_runs,
    )

    bank = generate_independent_bank(cfg, paths, logger)
    bank_augmented = add_marginal_features_confirm(
        bank,
        cfg,
        paths["bank"] / "candidate_metrics_with_marginals.csv",
        logger,
    )
    selected = select_independent_cohort(bank_augmented, cfg, paths, logger)
    selected_full = materialize_selected_hmms(selected, cfg, paths, logger)
    selected_full = add_selected_marginals_confirm(selected_full)
    atomic_write_csv(paths["selected"] / "confirmatory_selected_hmm_metrics_full.csv", selected_full)
    if len(selected_full) != expected_hmms:
        raise RuntimeError(f"Expected {expected_hmms} materialized HMMs; found {len(selected_full)}")

    dataset_paths = prepare_all_datasets(selected_full, cfg, paths, logger)
    preparation = {
        "status": "confirmatory_preparation_completed",
        "selected_hmms": int(len(selected_full)),
        "expected_neural_conditions": int(expected_runs),
        "root": str(paths["root"]),
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    atomic_write_json(paths["root"] / "CONFIRMATORY_PREPARATION_COMPLETE.json", preparation)
    if PREPARE_ONLY:
        logger.info("PREPARE_ONLY=True; stopping before training")
        return

    completed, expected = run_confirmatory_training_grid(
        selected_full, dataset_paths, cfg, paths, logger, device
    )
    if completed < expected:
        completion = {
            "status": "confirmatory_training_incomplete",
            "done_runs": int(completed),
            "expected_runs": int(expected),
            "remaining_runs": int(expected - completed),
            "root": str(paths["root"]),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        atomic_write_json(paths["root"] / "CONFIRMATORY_STATUS.json", completion)
        logger.info("Training is incomplete; rerun the same script to resume: %s", completion)
        return

    raw_results = collect_done_results(paths["runs"])
    atomic_write_csv(paths["tables"] / "confirmatory_training_raw.csv", raw_results)
    run_metrics = attach_recovery_metrics(raw_results, selected_full, cfg)
    atomic_write_csv(paths["tables"] / "confirmatory_training_with_metrics.csv", run_metrics)
    width = width_summary(run_metrics, cfg)
    width = width.merge(selected_full[["hmm_id", "K"]], on="hmm_id", how="left")
    atomic_write_csv(paths["tables"] / "confirmatory_width_summary.csv", width)

    ngram = pd.DataFrame()
    if RUN_NGRAM_BASELINE:
        ngram = run_ngram_baseline(selected_full, dataset_paths, cfg, paths, logger)
        ngram_neural_association(width, ngram, paths)

    predictions = pd.DataFrame()
    performance = pd.DataFrame()
    paired = pd.DataFrame()
    development_metrics = pd.DataFrame()
    if RUN_HOLDOUT_ANALYSIS or RUN_CROSS_K_CONFIRMATORY:
        development_metrics = locate_development_augmented_metrics()
    if RUN_HOLDOUT_ANALYSIS:
        development_width = build_matching_width_summary_from_v2(cfg)
        development_table = merged_outcome_feature_table(
            development_width, development_metrics, cfg
        )
        confirmatory_table = merged_outcome_feature_table(width, selected_full, cfg)
        atomic_write_csv(paths["tables"] / "development_feature_outcome_table.csv", development_table)
        atomic_write_csv(paths["tables"] / "confirmatory_feature_outcome_table.csv", confirmatory_table)
        predictions, performance, _ = fit_frozen_models_and_predict(
            development_table, confirmatory_table, cfg, paths
        )
        paired = paired_holdout_model_comparisons(predictions, cfg, paths)

    if RUN_CROSS_K_CONFIRMATORY:
        cross_pairs = make_cross_k_pairs(selected_full, development_metrics, cfg, paths)
        cross_k_confirmatory_test(cross_pairs, width, cfg, paths)

    create_confirmatory_figures(
        predictions, performance, width, selected_full, ngram, cfg, paths
    )

    completion = {
        "status": "confirmatory_pipeline_completed",
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "root": str(paths["root"]),
        "selected_hmms": int(len(selected_full)),
        "expected_runs": int(expected),
        "done_runs": int(len(raw_results)),
        "primary_outcomes": list(PRIMARY_OUTCOMES),
        "holdout_comparisons": int(len(paired)),
    }
    atomic_write_json(paths["root"] / "CONFIRMATORY_PIPELINE_COMPLETE.json", completion)
    logger.info("Independent 64-HMM confirmatory pipeline finished: %s", completion)



# =============================================================================
# Predictor-specific difficulty experiment
# =============================================================================
#
# Purpose
# -------
# This experiment tests whether count-based and neural sequence predictors
# induce different orderings of process difficulty.
#
# It selects 48 previously unused HMMs from the independent 4,000-HMM
# confirmatory candidate bank:
#
#   A. Context-occupancy contrast
#      - same K and closely matched predictive profile
#      - effective number of length-4 contexts differs by at least ~2x
#      - this contrast intentionally includes the accompanying marginal-entropy
#        change because H(U_4) and marginal entropy are tightly linked in this
#        generator. It is therefore called an occupancy contrast, not an
#        entropy-controlled causal contrast.
#
#   B. Predictive-gain contrast
#      - same K
#      - context occupancy, marginal entropy, rank, sigma, mixing, Bayes CE,
#        and total spectral signal are matched
#      - context gain differs by at least ~4x
#
# For each contrast and each K in {3,4,5,6}, three disjoint pairs are selected:
#
#   2 contrasts x 4 K values x 3 pairs = 24 pairs = 48 HMMs.
#
# Neural grid:
#   48 HMMs x 2 architectures x 3 widths x 5 seeds = 1,440 runs.
#
# Count-based grid:
#   n_train in {64,128,256,512,1024,2048}
#   additive smoothing alpha in {0.1,0.5,1.0}
#   interpolated backoff with the same alpha values and beta in {1,5,20}
#
# The script is resumable. Selection is locked before learner outcomes are
# computed. Completed neural conditions are skipped on rerun.
# =============================================================================


# -----------------------------------------------------------------------------
# User-editable paths and run controls
# -----------------------------------------------------------------------------

SOURCE_CONFIRM_ROOT = Path(
    "/content/drive/MyDrive/predictive_complexity_project/"
    "predictive_profile_confirmatory64_v1"
)

DIFFICULTY_ROOT = Path(
    "/content/drive/MyDrive/predictive_complexity_project/"
    "predictor_specific_difficulty_v1"
)

DIFFICULTY_MASTER_SEED = 20260718

K_VALUES_DIFFICULTY = (3, 4, 5, 6)
PAIRS_PER_K_PER_CONTRAST = 3
DIFFICULTY_ARCHITECTURES = ("GRU", "Transformer")
DIFFICULTY_DIMENSIONS = (8, 32, 128)
DIFFICULTY_SEEDS = (1, 2, 3, 4, 5)

NEURAL_N_TRAIN = 256
NGRAM_TRAIN_SIZES = (64, 128, 256, 512, 1024, 2048)
NGRAM_ALPHAS = (0.1, 0.5, 1.0)
NGRAM_BACKOFF_BETAS = (1.0, 5.0, 20.0)

PRIMARY_NGRAM_ALPHA = 0.5
PRIMARY_BACKOFF_BETA = 5.0
BOOTSTRAP_REPEATS_DIFFICULTY = 10000
PERMUTATION_REPEATS_DIFFICULTY = 10000

PREPARE_ONLY_DIFFICULTY = False
RUN_NGRAM_DIFFICULTY = True
RUN_NEURAL_DIFFICULTY = True
RUN_ANALYSIS_DIFFICULTY = True

FORCE_RESELECT_DIFFICULTY = False
FORCE_RETRAIN_DIFFICULTY = False

# Set to an integer such as 100 to run that many new neural conditions and stop
# cleanly. None runs all remaining conditions.
MAX_NEW_NEURAL_CONDITIONS_PER_EXECUTION: Optional[int] = None

# Tiny end-to-end test. Use only before the full run and return it to False.
DIFFICULTY_SMOKE_TEST = os.environ.get("PP_DIFFICULTY_SMOKE_TEST", "0") == "1"


# -----------------------------------------------------------------------------
# Fixed matching protocol
# -----------------------------------------------------------------------------

# The first tier is the planned protocol. Later tiers are conservative fallbacks
# used only if a K-specific candidate pool cannot supply three disjoint pairs.
OCCUPANCY_MATCH_TIERS: Tuple[Dict[str, float], ...] = (
    {
        "min_occupancy_ratio": 2.0,
        "max_gain_ratio": 1.50,
        "max_sigma_ratio": 1.50,
        "max_tau_ratio": 1.50,
        "max_bayes_ratio": 1.15,
        "max_signal_ratio": 2.00,
        "max_minprob_ratio": 2.00,
    },
    {
        "min_occupancy_ratio": 1.8,
        "max_gain_ratio": 1.60,
        "max_sigma_ratio": 1.60,
        "max_tau_ratio": 1.60,
        "max_bayes_ratio": 1.20,
        "max_signal_ratio": 2.50,
        "max_minprob_ratio": 3.00,
    },
)

GAIN_MATCH_TIERS: Tuple[Dict[str, float], ...] = (
    {
        "min_gain_ratio": 4.0,
        "max_occupancy_ratio": 1.25,
        "max_entropy_difference": 0.05,
        "max_minprob_ratio": 2.00,
        "max_sigma_ratio": 1.50,
        "max_tau_ratio": 1.50,
        "max_bayes_ratio": 1.15,
        "max_signal_ratio": 2.00,
    },
    {
        "min_gain_ratio": 3.0,
        "max_occupancy_ratio": 1.35,
        "max_entropy_difference": 0.08,
        "max_minprob_ratio": 3.00,
        "max_sigma_ratio": 1.60,
        "max_tau_ratio": 1.60,
        "max_bayes_ratio": 1.20,
        "max_signal_ratio": 2.50,
    },
)


# -----------------------------------------------------------------------------
# Configuration and source paths
# -----------------------------------------------------------------------------

def make_difficulty_config() -> Config:
    cfg = Config()
    cfg.PROJECT_ROOT = str(DIFFICULTY_ROOT)
    cfg.MASTER_SEED = DIFFICULTY_MASTER_SEED
    cfg.IMPORT_EXISTING_HMMS = False
    cfg.GENERATE_BANK = False
    cfg.RUN_SAMPLE_SIZE_EXPERIMENT = False
    cfg.RUN_EMPIRICAL_RANK = False
    cfg.RUN_TRAINING = RUN_NEURAL_DIFFICULTY
    cfg.RUN_ANALYSIS = RUN_ANALYSIS_DIFFICULTY
    cfg.K_VALUES = K_VALUES_DIFFICULTY
    cfg.ARCHITECTURES = DIFFICULTY_ARCHITECTURES
    cfg.DIMENSIONS = DIFFICULTY_DIMENSIONS
    cfg.SEEDS = DIFFICULTY_SEEDS
    cfg.N_TRAIN_MAX = max(NGRAM_TRAIN_SIZES)
    cfg.N_TRAIN_MAIN = NEURAL_N_TRAIN
    cfg.N_VAL = 32
    cfg.N_TEST = 64
    cfg.SEQUENCE_LENGTH = 300
    cfg.EVAL_WINDOWS_PER_SEQUENCE = 8
    cfg.MAX_EPOCHS = 48
    cfg.MIN_EPOCHS = 12
    cfg.EARLY_STOP_PATIENCE = 8
    cfg.BATCH_SIZE = 16
    cfg.CONTINUE_AFTER_ERROR = True
    cfg.RETRY_FAILED_CONDITIONS = True
    cfg.FORCE_RETRAIN = FORCE_RETRAIN_DIFFICULTY
    cfg.MATCH_PROTOCOL_VERSION = "predictor_specific_difficulty_matched_v1"

    if DIFFICULTY_SMOKE_TEST:
        cfg.K_VALUES = (3,)
        cfg.ARCHITECTURES = ("GRU",)
        cfg.DIMENSIONS = (8,)
        cfg.SEEDS = (1,)
        cfg.N_TRAIN_MAX = 64
        cfg.N_TRAIN_MAIN = 16
        cfg.N_VAL = 8
        cfg.N_TEST = 8
        cfg.SEQUENCE_LENGTH = 40
        cfg.EVAL_WINDOWS_PER_SEQUENCE = 2
        cfg.MAX_EPOCHS = 2
        cfg.MIN_EPOCHS = 1
        cfg.EARLY_STOP_PATIENCE = 1
        cfg.BATCH_SIZE = 8
    return cfg


def source_candidate_path() -> Path:
    candidates = [
        SOURCE_CONFIRM_ROOT / "candidate_bank" / "candidate_metrics_with_marginals.csv",
        SOURCE_CONFIRM_ROOT / "candidate_bank" / "candidate_bank_metrics.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "The confirmatory candidate bank was not found. Preserve and check:\n"
        f"  {SOURCE_CONFIRM_ROOT / 'candidate_bank'}"
    )


def source_used_hmm_ids() -> set[str]:
    candidates = [
        SOURCE_CONFIRM_ROOT / "selected_hmms" / "confirmatory_selected_hmms.csv",
        SOURCE_CONFIRM_ROOT / "selected_hmms" / "confirmatory_selected_hmm_metrics_full.csv",
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if "hmm_id" not in df.columns:
                continue
            return set(df["hmm_id"].astype(str))
    raise FileNotFoundError(
        "Could not identify the 64 HMMs used in the previous confirmatory study."
    )


# -----------------------------------------------------------------------------
# Exact context-occupancy features
# -----------------------------------------------------------------------------

def context_occupancy_features(
    T: np.ndarray,
    O: np.ndarray,
    pi: np.ndarray,
    max_context: int,
) -> Dict[str, float]:
    out: Dict[str, float] = {}
    for h in range(1, max_context + 1):
        masses = context_forward_masses(T, O, pi, h)
        probabilities = masses.sum(axis=1)
        probabilities = probabilities / np.clip(probabilities.sum(), 1e-15, None)
        positive = probabilities > 0
        entropy = float(
            -np.sum(
                probabilities[positive]
                * np.log(np.clip(probabilities[positive], 1e-15, None))
            )
        )
        out[f"context_entropy_h{h}"] = entropy
        out[f"context_effective_count_h{h}"] = float(np.exp(entropy))
        out[f"context_support_h{h}"] = int(np.sum(positive))
        out[f"context_max_prob_h{h}"] = float(probabilities.max())
    return out


def ensure_bank_marginals_and_context(
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> pd.DataFrame:
    output = paths["bank"] / "candidate_metrics_with_context.csv"
    source = source_candidate_path()
    bank = pd.read_csv(source)

    required_marginals = {
        "symbol_entropy",
        "min_symbol_prob_recomputed",
    }
    required_context = {
        "context_entropy_h4",
        "context_effective_count_h4",
    }

    if output.exists():
        cached = pd.read_csv(output)
        if (
            len(cached) == len(bank)
            and set(cached["hmm_id"].astype(str)) == set(bank["hmm_id"].astype(str))
            and required_marginals.union(required_context).issubset(cached.columns)
        ):
            logger.info("Reusing candidate context-feature cache: %s", output)
            return cached

    rows: List[Dict[str, Any]] = []
    for index, (_, row) in enumerate(bank.iterrows(), start=1):
        T, O, pi = regenerate_from_row(row, cfg)
        merged = row.to_dict()
        if not required_marginals.issubset(bank.columns):
            merged.update(symbol_marginal_features_confirm(T, O, pi))
        merged.update(context_occupancy_features(T, O, pi, cfg.MAX_CONTEXT))
        rows.append(merged)
        if index % 250 == 0:
            logger.info("Computed context features for %d/%d candidates", index, len(bank))

    out = pd.DataFrame(rows)
    atomic_write_csv(output, out)
    return out


# -----------------------------------------------------------------------------
# Matched-pair selection
# -----------------------------------------------------------------------------

def finite_ratio(a: float, b: float, eps: float = 1e-15) -> float:
    a = max(float(a), eps)
    b = max(float(b), eps)
    return max(a, b) / min(a, b)


def pair_columns(cfg: Config) -> Dict[str, str]:
    m = cfg.SELECTION_FUTURE_LENGTH
    return {
        "rank": f"pred_h{cfg.MAX_CONTEXT}_m{m}_r95",
        "sigma": f"pred_h{cfg.MAX_CONTEXT}_m{m}_sigma95",
        "signal": f"pred_h{cfg.MAX_CONTEXT}_m{m}_signal",
    }


def enumerate_pair_candidates(
    sub: pd.DataFrame,
    contrast: str,
    tier: Mapping[str, float],
    tier_index: int,
    cfg: Config,
) -> List[Dict[str, Any]]:
    cols = pair_columns(cfg)
    records = sub.reset_index(drop=True)
    n = len(records)
    if n < 2:
        return []

    i_idx, j_idx = np.triu_indices(n, 1)
    arrays = {
        "rank": records[cols["rank"]].to_numpy(float),
        "sigma": records[cols["sigma"]].to_numpy(float),
        "signal": records[cols["signal"]].to_numpy(float),
        "gain": records["context_gain_area"].to_numpy(float),
        "tau": records["tau_observed_proxy"].to_numpy(float),
        "bayes": records[f"bayes_ce_h{cfg.MAX_CONTEXT}"].to_numpy(float),
        "occupancy": records[f"context_effective_count_h{cfg.MAX_CONTEXT}"].to_numpy(float),
        "entropy": records["symbol_entropy"].to_numpy(float),
        "minprob": records["min_symbol_prob_recomputed"].to_numpy(float),
    }

    keep = arrays["rank"][i_idx].astype(int) == arrays["rank"][j_idx].astype(int)

    def vector_ratio(name: str) -> np.ndarray:
        left = np.clip(arrays[name][i_idx], 1e-15, None)
        right = np.clip(arrays[name][j_idx], 1e-15, None)
        return np.maximum(left, right) / np.minimum(left, right)

    gain_ratio = vector_ratio("gain")
    sigma_ratio = vector_ratio("sigma")
    tau_ratio = vector_ratio("tau")
    bayes_ratio = vector_ratio("bayes")
    signal_ratio = vector_ratio("signal")
    occupancy_ratio = vector_ratio("occupancy")
    minprob_ratio = vector_ratio("minprob")
    entropy_difference = np.abs(arrays["entropy"][i_idx] - arrays["entropy"][j_idx])

    if contrast == "occupancy_contrast":
        contrast_ratio = occupancy_ratio
        keep &= contrast_ratio >= float(tier["min_occupancy_ratio"])
        keep &= gain_ratio <= float(tier["max_gain_ratio"])
        keep &= sigma_ratio <= float(tier["max_sigma_ratio"])
        keep &= tau_ratio <= float(tier["max_tau_ratio"])
        keep &= bayes_ratio <= float(tier["max_bayes_ratio"])
        keep &= signal_ratio <= float(tier["max_signal_ratio"])
        keep &= minprob_ratio <= float(tier["max_minprob_ratio"])
        control_distance = np.sqrt(
            np.log(gain_ratio) ** 2
            + np.log(sigma_ratio) ** 2
            + np.log(tau_ratio) ** 2
            + np.log(bayes_ratio) ** 2
            + np.log(signal_ratio) ** 2
            + np.log(minprob_ratio) ** 2
        )
    elif contrast == "gain_contrast":
        contrast_ratio = gain_ratio
        keep &= contrast_ratio >= float(tier["min_gain_ratio"])
        keep &= occupancy_ratio <= float(tier["max_occupancy_ratio"])
        keep &= entropy_difference <= float(tier["max_entropy_difference"])
        keep &= minprob_ratio <= float(tier["max_minprob_ratio"])
        keep &= sigma_ratio <= float(tier["max_sigma_ratio"])
        keep &= tau_ratio <= float(tier["max_tau_ratio"])
        keep &= bayes_ratio <= float(tier["max_bayes_ratio"])
        keep &= signal_ratio <= float(tier["max_signal_ratio"])
        control_distance = np.sqrt(
            np.log(occupancy_ratio) ** 2
            + (entropy_difference / max(float(tier["max_entropy_difference"]), 1e-12)) ** 2
            + np.log(minprob_ratio) ** 2
            + np.log(sigma_ratio) ** 2
            + np.log(tau_ratio) ** 2
            + np.log(bayes_ratio) ** 2
            + np.log(signal_ratio) ** 2
        )
    else:
        raise ValueError(f"Unknown contrast: {contrast}")

    score = 3.0 * np.log(contrast_ratio) - control_distance - 0.25 * tier_index
    accepted = np.flatnonzero(keep)

    candidates: List[Dict[str, Any]] = []
    for position in accepted:
        a = records.iloc[int(i_idx[position])]
        b = records.iloc[int(j_idx[position])]

        if contrast == "occupancy_contrast":
            high_is_a = float(a[f"context_effective_count_h{cfg.MAX_CONTEXT}"]) >= float(
                b[f"context_effective_count_h{cfg.MAX_CONTEXT}"]
            )
        else:
            high_is_a = float(a["context_gain_area"]) >= float(b["context_gain_area"])

        low = b if high_is_a else a
        high = a if high_is_a else b

        candidates.append(
            {
                "contrast": contrast,
                "hmm_low": str(low["hmm_id"]),
                "hmm_high": str(high["hmm_id"]),
                "K": int(a["K"]),
                "selection_score": float(score[position]),
                "tier_index": int(tier_index),
                "contrast_ratio": float(contrast_ratio[position]),
                "control_distance": float(control_distance[position]),
                "rank_low": int(low[cols["rank"]]),
                "rank_high": int(high[cols["rank"]]),
                "gain_low": float(low["context_gain_area"]),
                "gain_high": float(high["context_gain_area"]),
                "gain_ratio": float(gain_ratio[position]),
                "occupancy_low": float(low[f"context_effective_count_h{cfg.MAX_CONTEXT}"]),
                "occupancy_high": float(high[f"context_effective_count_h{cfg.MAX_CONTEXT}"]),
                "occupancy_ratio": float(occupancy_ratio[position]),
                "entropy_low": float(low["symbol_entropy"]),
                "entropy_high": float(high["symbol_entropy"]),
                "entropy_difference": float(entropy_difference[position]),
                "minprob_low": float(low["min_symbol_prob_recomputed"]),
                "minprob_high": float(high["min_symbol_prob_recomputed"]),
                "minprob_ratio": float(minprob_ratio[position]),
                "sigma_low": float(low[cols["sigma"]]),
                "sigma_high": float(high[cols["sigma"]]),
                "sigma_ratio": float(sigma_ratio[position]),
                "tau_low": float(low["tau_observed_proxy"]),
                "tau_high": float(high["tau_observed_proxy"]),
                "tau_ratio": float(tau_ratio[position]),
                "bayes_ce_low": float(low[f"bayes_ce_h{cfg.MAX_CONTEXT}"]),
                "bayes_ce_high": float(high[f"bayes_ce_h{cfg.MAX_CONTEXT}"]),
                "bayes_ce_ratio": float(bayes_ratio[position]),
                "signal_low": float(low[cols["signal"]]),
                "signal_high": float(high[cols["signal"]]),
                "signal_ratio": float(signal_ratio[position]),
                "matching_protocol": cfg.MATCH_PROTOCOL_VERSION,
            }
        )

    return sorted(candidates, key=lambda row: row["selection_score"], reverse=True)


def select_difficulty_pairs(
    bank: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    selected_path = paths["selected"] / "difficulty_selected_hmms.csv"
    pairs_path = paths["selected"] / "difficulty_matched_pairs.csv"
    lock_path = paths["selected"] / "SELECTION_LOCK.json"

    if (
        not FORCE_RESELECT_DIFFICULTY
        and selected_path.exists()
        and pairs_path.exists()
        and lock_path.exists()
    ):
        selected = pd.read_csv(selected_path)
        pairs = pd.read_csv(pairs_path)
        expected_pairs = (
            2 * len(cfg.K_VALUES) * (1 if DIFFICULTY_SMOKE_TEST else PAIRS_PER_K_PER_CONTRAST)
        )
        if len(pairs) != expected_pairs:
            raise RuntimeError(
                f"Locked pair file has {len(pairs)} rows, expected {expected_pairs}."
            )
        logger.info("Reusing locked predictor-specific selection")
        return selected, pairs

    used = source_used_hmm_ids()
    pool = bank[
        bank["K"].astype(int).isin([int(x) for x in cfg.K_VALUES])
        & ~bank["hmm_id"].astype(str).isin(used)
    ].copy()

    required = {
        "context_effective_count_h4",
        "symbol_entropy",
        "min_symbol_prob_recomputed",
        "context_gain_area",
        "tau_observed_proxy",
        f"bayes_ce_h{cfg.MAX_CONTEXT}",
    }.union(pair_columns(cfg).values())
    missing = sorted(required.difference(pool.columns))
    if missing:
        raise RuntimeError(f"Candidate bank lacks required matching columns: {missing}")

    pairs_per_cell = 1 if DIFFICULTY_SMOKE_TEST else PAIRS_PER_K_PER_CONTRAST
    globally_used = set(used)
    chosen_rows: List[Dict[str, Any]] = []

    contrast_tiers = (
        ("occupancy_contrast", OCCUPANCY_MATCH_TIERS),
        ("gain_contrast", GAIN_MATCH_TIERS),
    )

    for contrast, tiers in contrast_tiers:
        for K in cfg.K_VALUES:
            available = pool[
                (pool["K"].astype(int) == int(K))
                & ~pool["hmm_id"].astype(str).isin(globally_used)
            ].copy()

            selected_here: List[Dict[str, Any]] = []
            local_used: set[str] = set()

            for tier_index, tier in enumerate(tiers):
                candidates = enumerate_pair_candidates(
                    available, contrast, tier, tier_index, cfg
                )
                for candidate in candidates:
                    a = str(candidate["hmm_low"])
                    b = str(candidate["hmm_high"])
                    if a in local_used or b in local_used:
                        continue
                    candidate["tier_parameters"] = json.dumps(dict(tier), sort_keys=True)
                    selected_here.append(candidate)
                    local_used.update((a, b))
                    if len(selected_here) >= pairs_per_cell:
                        break
                if len(selected_here) >= pairs_per_cell:
                    break

            if len(selected_here) < pairs_per_cell:
                raise RuntimeError(
                    f"Could select only {len(selected_here)} {contrast} pairs for K={K}; "
                    f"expected {pairs_per_cell}."
                )

            for row in selected_here:
                row["pair_id"] = (
                    f"{contrast.replace('_contrast', '')}_K{K}_"
                    f"p{len(chosen_rows) + 1:02d}"
                )
                chosen_rows.append(row)
                globally_used.update((str(row["hmm_low"]), str(row["hmm_high"])))

            logger.info(
                "Selected %d %s pairs for K=%s", len(selected_here), contrast, K
            )

    pairs = pd.DataFrame(chosen_rows)
    selected_ids: List[str] = []
    membership: Dict[str, Dict[str, Any]] = {}
    for _, pair in pairs.iterrows():
        for role, column in (("low", "hmm_low"), ("high", "hmm_high")):
            hmm_id = str(pair[column])
            selected_ids.append(hmm_id)
            membership[hmm_id] = {
                "pair_id": str(pair["pair_id"]),
                "contrast": str(pair["contrast"]),
                "pair_role": role,
            }

    selected_ids = list(dict.fromkeys(selected_ids))
    expected_hmms = 2 * len(pairs)
    if len(selected_ids) != expected_hmms:
        raise RuntimeError(
            "The selected matched pairs are not globally disjoint: "
            f"{len(selected_ids)} unique HMMs for {len(pairs)} pairs."
        )

    selected = bank[bank["hmm_id"].astype(str).isin(selected_ids)].copy()
    selected["selection_order"] = selected["hmm_id"].astype(str).map(
        {hmm_id: index for index, hmm_id in enumerate(selected_ids)}
    )
    selected["pair_id"] = selected["hmm_id"].astype(str).map(
        {key: value["pair_id"] for key, value in membership.items()}
    )
    selected["contrast"] = selected["hmm_id"].astype(str).map(
        {key: value["contrast"] for key, value in membership.items()}
    )
    selected["pair_role"] = selected["hmm_id"].astype(str).map(
        {key: value["pair_role"] for key, value in membership.items()}
    )
    selected = selected.sort_values("selection_order")

    atomic_write_csv(selected_path, selected)
    atomic_write_csv(pairs_path, pairs)
    atomic_write_json(
        lock_path,
        {
            "status": "selection_locked",
            "protocol": cfg.MATCH_PROTOCOL_VERSION,
            "selected_hmms": int(len(selected)),
            "pairs": int(len(pairs)),
            "source_bank": str(source_candidate_path()),
            "excluded_previous_confirmatory_hmms": int(len(used)),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "learner_outcomes_used_for_selection": False,
        },
    )
    logger.info("Locked %d pairs and %d HMMs", len(pairs), len(selected))
    return selected, pairs


def add_selected_context_and_marginals(
    selected_full: pd.DataFrame,
    selected_design: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    design_columns = ["hmm_id", "pair_id", "contrast", "pair_role", "selection_order"]
    design = selected_design[design_columns].drop_duplicates("hmm_id")
    merged = selected_full.drop(
        columns=[column for column in design_columns[1:] if column in selected_full.columns],
        errors="ignore",
    ).merge(design, on="hmm_id", how="left")

    rows: List[Dict[str, Any]] = []
    for _, row in merged.iterrows():
        with np.load(Path(str(row["hmm_file"]))) as data:
            T, O, pi = data["T"], data["O"], data["pi"]
        rows.append(
            {
                **row.to_dict(),
                **symbol_marginal_features_confirm(T, O, pi),
                **context_occupancy_features(T, O, pi, cfg.MAX_CONTEXT),
            }
        )
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# Neural training grid with resumable chunking
# -----------------------------------------------------------------------------

def run_difficulty_neural_grid(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
    device: torch.device,
) -> Tuple[int, int]:
    expected = (
        len(selected_full)
        * len(cfg.ARCHITECTURES)
        * len(cfg.DIMENSIONS)
        * len(cfg.SEEDS)
    )
    completed_before = len(collect_done_results(paths["runs"]))

    conditions: List[Tuple[str, str, int, int]] = []
    for _, row in selected_full.iterrows():
        hmm_id = str(row["hmm_id"])
        for architecture in cfg.ARCHITECTURES:
            for dimension in cfg.DIMENSIONS:
                for seed in cfg.SEEDS:
                    run_dir = condition_directory(
                        paths["runs"],
                        hmm_id,
                        architecture,
                        int(dimension),
                        int(seed),
                        cfg.N_TRAIN_MAIN,
                    )
                    if (run_dir / "DONE.json").exists() and not cfg.FORCE_RETRAIN:
                        continue
                    conditions.append((hmm_id, architecture, int(dimension), int(seed)))

    if MAX_NEW_NEURAL_CONDITIONS_PER_EXECUTION is not None:
        conditions = conditions[: int(MAX_NEW_NEURAL_CONDITIONS_PER_EXECUTION)]

    logger.info(
        "Neural grid expected=%d completed_before=%d scheduled_now=%d",
        expected,
        completed_before,
        len(conditions),
    )

    for index, (hmm_id, architecture, dimension, seed) in enumerate(conditions, start=1):
        if hmm_id not in dataset_paths:
            continue
        logger.info(
            "Difficulty condition %d/%d: %s %s d=%d seed=%d",
            index,
            len(conditions),
            hmm_id,
            architecture,
            dimension,
            seed,
        )
        try:
            train_one_condition(
                hmm_id=hmm_id,
                architecture=architecture,
                dimension=dimension,
                seed=seed,
                n_train=cfg.N_TRAIN_MAIN,
                dataset_path=dataset_paths[hmm_id],
                run_base=paths["runs"],
                cfg=cfg,
                device=device,
                logger=logger,
            )
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            record_failure(
                paths,
                "difficulty_training",
                f"{hmm_id}_{architecture}_d{dimension}_s{seed}",
                exc,
            )
            logger.error("Difficulty condition failed: %s", exc)
            if not cfg.CONTINUE_AFTER_ERROR:
                raise

    completed_after = len(collect_done_results(paths["runs"]))
    return completed_after, expected


# -----------------------------------------------------------------------------
# Efficient count-based predictors
# -----------------------------------------------------------------------------

def encoded_contexts_and_targets(
    sequences: np.ndarray,
    context_length: int,
    vocab_size: int,
    evaluation_start: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    sequences = np.asarray(sequences, dtype=np.int64)
    n_sequences, length = sequences.shape
    start = context_length if evaluation_start is None else int(evaluation_start)
    if start < context_length:
        raise ValueError("evaluation_start must be at least context_length")
    if start >= length:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)

    width = length - start
    codes = np.zeros((n_sequences, width), dtype=np.int64)
    for offset in range(context_length):
        left = start - context_length + offset
        right = length - context_length + offset
        codes = codes * vocab_size + sequences[:, left:right]
    targets = sequences[:, start:length]
    return codes.reshape(-1), targets.reshape(-1)


def fit_ngram_array_counts(
    train_sequences: np.ndarray,
    max_context: int,
    vocab_size: int,
) -> Tuple[np.ndarray, Dict[int, np.ndarray]]:
    unigram = np.bincount(
        np.asarray(train_sequences, dtype=np.int64).reshape(-1),
        minlength=vocab_size,
    ).astype(np.int64)

    tables: Dict[int, np.ndarray] = {}
    for h in range(1, max_context + 1):
        codes, targets = encoded_contexts_and_targets(
            train_sequences, h, vocab_size
        )
        flat = codes * vocab_size + targets
        table = np.bincount(
            flat,
            minlength=(vocab_size ** h) * vocab_size,
        ).reshape(vocab_size ** h, vocab_size)
        tables[h] = table.astype(np.int64)
    return unigram, tables


def empirical_effective_context_count(table: np.ndarray) -> float:
    totals = np.asarray(table.sum(axis=1), dtype=np.float64)
    positive = totals > 0
    if not np.any(positive):
        return 0.0
    probabilities = totals[positive] / totals[positive].sum()
    entropy = -np.sum(probabilities * np.log(np.clip(probabilities, 1e-15, None)))
    return float(np.exp(entropy))


def additive_target_probabilities(
    table: np.ndarray,
    codes: np.ndarray,
    targets: np.ndarray,
    alpha: float,
    vocab_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    totals = table.sum(axis=1)
    context_totals = totals[codes].astype(np.float64)
    target_counts = table[codes, targets].astype(np.float64)
    probabilities = (target_counts + alpha) / (
        context_totals + vocab_size * alpha
    )
    return probabilities, context_totals


def interpolated_backoff_target_probabilities(
    unigram: np.ndarray,
    tables: Dict[int, np.ndarray],
    test_sequences: np.ndarray,
    h: int,
    alpha: float,
    beta: float,
    vocab_size: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    _, targets = encoded_contexts_and_targets(
        test_sequences, h, vocab_size, evaluation_start=h
    )
    unigram_probabilities = (unigram.astype(np.float64) + alpha) / (
        float(unigram.sum()) + vocab_size * alpha
    )
    probability = unigram_probabilities[targets]
    direct_totals = np.zeros_like(probability)
    cumulative_backoff_weight = np.zeros_like(probability)

    for level in range(1, h + 1):
        codes, aligned_targets = encoded_contexts_and_targets(
            test_sequences,
            level,
            vocab_size,
            evaluation_start=h,
        )
        if not np.array_equal(aligned_targets, targets):
            raise AssertionError("Backoff target alignment failed")
        table = tables[level]
        totals = table.sum(axis=1)
        context_totals = totals[codes].astype(np.float64)
        direct = (table[codes, targets].astype(np.float64) + alpha) / (
            context_totals + vocab_size * alpha
        )
        lam = context_totals / (context_totals + beta)
        probability = lam * direct + (1.0 - lam) * probability
        cumulative_backoff_weight = 1.0 - lam
        if level == h:
            direct_totals = context_totals

    return probability, direct_totals, cumulative_backoff_weight


def evaluate_count_predictor_curve(
    train_sequences: np.ndarray,
    test_sequences: np.ndarray,
    cfg: Config,
    alpha: float,
    method: str,
    beta: Optional[float],
) -> Tuple[Dict[int, float], List[Dict[str, Any]]]:
    unigram, tables = fit_ngram_array_counts(
        train_sequences, cfg.MAX_CONTEXT, cfg.VOCAB_SIZE
    )
    curve: Dict[int, float] = {}
    diagnostics: List[Dict[str, Any]] = []

    for h in range(1, cfg.MAX_CONTEXT + 1):
        codes, targets = encoded_contexts_and_targets(
            test_sequences, h, cfg.VOCAB_SIZE
        )
        table = tables[h]
        if method == "additive":
            probabilities, context_totals = additive_target_probabilities(
                table, codes, targets, alpha, cfg.VOCAB_SIZE
            )
            backoff_weight = np.zeros_like(probabilities)
        elif method == "interpolated_backoff":
            if beta is None:
                raise ValueError("beta is required for interpolated_backoff")
            probabilities, context_totals, backoff_weight = (
                interpolated_backoff_target_probabilities(
                    unigram,
                    tables,
                    test_sequences,
                    h,
                    alpha,
                    float(beta),
                    cfg.VOCAB_SIZE,
                )
            )
        else:
            raise ValueError(f"Unknown count predictor method: {method}")

        curve[h] = float(
            -np.mean(np.log(np.clip(probabilities, 1e-15, 1.0)))
        )

        positive_test_totals = context_totals[context_totals > 0]
        diagnostics.append(
            {
                "h": int(h),
                "method": method,
                "alpha": float(alpha),
                "beta": np.nan if beta is None else float(beta),
                "test_windows": int(len(targets)),
                "observed_train_contexts": int(np.sum(table.sum(axis=1) > 0)),
                "possible_contexts": int(cfg.VOCAB_SIZE ** h),
                "unseen_test_context_rate": float(np.mean(context_totals == 0)),
                "mean_train_count_for_test_context": float(np.mean(context_totals)),
                "median_train_count_for_test_context": float(np.median(context_totals)),
                "mean_positive_train_count_for_test_context": (
                    float(np.mean(positive_test_totals))
                    if len(positive_test_totals)
                    else 0.0
                ),
                "empirical_effective_context_count": empirical_effective_context_count(
                    table
                ),
                "mean_backoff_weight": float(np.mean(backoff_weight)),
            }
        )
    return curve, diagnostics


def run_count_predictor_experiment(
    selected_full: pd.DataFrame,
    dataset_paths: Dict[str, Path],
    cfg: Config,
    paths: Dict[str, Path],
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    summary_path = paths["tables"] / "count_predictor_summary.csv"
    diagnostics_path = paths["tables"] / "count_predictor_context_diagnostics.csv"

    if summary_path.exists() and diagnostics_path.exists():
        summary = pd.read_csv(summary_path)
        diagnostics = pd.read_csv(diagnostics_path)
        expected_methods = (
            len(NGRAM_ALPHAS)
            + len(NGRAM_ALPHAS) * len(NGRAM_BACKOFF_BETAS)
        )
        expected_summary = len(selected_full) * len(NGRAM_TRAIN_SIZES) * expected_methods
        if len(summary) == expected_summary:
            logger.info("Reusing completed count-predictor experiment")
            return summary, diagnostics

    metric_lookup = selected_full.set_index("hmm_id", drop=False).to_dict("index")
    summary_rows: List[Dict[str, Any]] = []
    diagnostic_rows: List[Dict[str, Any]] = []

    methods: List[Tuple[str, float, Optional[float]]] = []
    for alpha in NGRAM_ALPHAS:
        methods.append(("additive", float(alpha), None))
        for beta in NGRAM_BACKOFF_BETAS:
            methods.append(("interpolated_backoff", float(alpha), float(beta)))

    for hmm_index, (hmm_id, dataset_path) in enumerate(dataset_paths.items(), start=1):
        data = load_dataset(dataset_path)
        test = data["test"][: cfg.N_TEST]
        metric = metric_lookup[hmm_id]
        H = cfg.MAX_CONTEXT

        bayes_curve = {
            h: float(metric[f"bayes_ce_h{h}"])
            for h in range(1, H + 1)
        }
        bayes_shape = np.array(
            [bayes_curve[h] - bayes_curve[H] for h in range(1, H + 1)]
        )
        bayes_gap = bayes_curve[1] - bayes_curve[H]

        for n_train in NGRAM_TRAIN_SIZES:
            if n_train > len(data["train"]):
                continue
            train = data["train"][: int(n_train)]

            for method, alpha, beta in methods:
                curve, diagnostics = evaluate_count_predictor_curve(
                    train,
                    test,
                    cfg,
                    alpha=alpha,
                    method=method,
                    beta=beta,
                )
                model_shape = np.array(
                    [curve[h] - curve[H] for h in range(1, H + 1)]
                )
                model_gap = curve[1] - curve[H]

                row: Dict[str, Any] = {
                    "hmm_id": hmm_id,
                    "K": int(metric["K"]),
                    "pair_id": metric.get("pair_id", ""),
                    "contrast": metric.get("contrast", ""),
                    "pair_role": metric.get("pair_role", ""),
                    "n_train": int(n_train),
                    "method": method,
                    "alpha": float(alpha),
                    "beta": np.nan if beta is None else float(beta),
                    "shape_rmse": float(
                        np.sqrt(np.mean((model_shape - bayes_shape) ** 2))
                    ),
                    "excess_ce_H": float(curve[H] - bayes_curve[H]),
                    "recovery_ratio": (
                        float(model_gap / bayes_gap)
                        if abs(bayes_gap) > 1e-12
                        else np.nan
                    ),
                }
                for h in range(1, H + 1):
                    row[f"test_ce_h{h}"] = float(curve[h])
                    row[f"bayes_ce_h{h}"] = float(bayes_curve[h])
                summary_rows.append(row)

                for diagnostic in diagnostics:
                    diagnostic_rows.append(
                        {
                            "hmm_id": hmm_id,
                            "K": int(metric["K"]),
                            "pair_id": metric.get("pair_id", ""),
                            "contrast": metric.get("contrast", ""),
                            "pair_role": metric.get("pair_role", ""),
                            "n_train": int(n_train),
                            **diagnostic,
                        }
                    )

        if hmm_index % 4 == 0:
            logger.info(
                "Count-predictor experiment completed for %d/%d HMMs",
                hmm_index,
                len(dataset_paths),
            )
            atomic_write_csv(summary_path, pd.DataFrame(summary_rows))
            atomic_write_csv(diagnostics_path, pd.DataFrame(diagnostic_rows))

    summary = pd.DataFrame(summary_rows)
    diagnostics = pd.DataFrame(diagnostic_rows)
    atomic_write_csv(summary_path, summary)
    atomic_write_csv(diagnostics_path, diagnostics)
    return summary, diagnostics


# -----------------------------------------------------------------------------
# Pair-level analyses
# -----------------------------------------------------------------------------

def pair_effect_rows(
    outcomes: pd.DataFrame,
    pairs: pd.DataFrame,
    value_columns: Sequence[str],
    grouping_columns: Sequence[str],
    source: str,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for _, pair in pairs.iterrows():
        low_id = str(pair["hmm_low"])
        high_id = str(pair["hmm_high"])
        contrast = str(pair["contrast"])

        low = outcomes[outcomes["hmm_id"].astype(str) == low_id]
        high = outcomes[outcomes["hmm_id"].astype(str) == high_id]
        if low.empty or high.empty:
            continue

        merge_columns = list(grouping_columns)
        low_keep = merge_columns + list(value_columns)
        high_keep = merge_columns + list(value_columns)
        merged = low[low_keep].merge(
            high[high_keep],
            on=merge_columns,
            suffixes=("_low", "_high"),
        )

        for _, row in merged.iterrows():
            base: Dict[str, Any] = {
                "source": source,
                "pair_id": str(pair["pair_id"]),
                "contrast": contrast,
                "K": int(pair["K"]),
                "hmm_low": low_id,
                "hmm_high": high_id,
            }
            for column in merge_columns:
                base[column] = row[column]

            for value in value_columns:
                rows.append(
                    {
                        **base,
                        "outcome": value,
                        "value_low": float(row[f"{value}_low"]),
                        "value_high": float(row[f"{value}_high"]),
                        "effect_high_minus_low": float(
                            row[f"{value}_high"] - row[f"{value}_low"]
                        ),
                    }
                )
    return pd.DataFrame(rows)


def bootstrap_signflip_summary(
    effects: pd.DataFrame,
    group_columns: Sequence[str],
    cfg: Config,
) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.MASTER_SEED + 991)
    rows: List[Dict[str, Any]] = []

    for keys, group in effects.groupby(list(group_columns), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        values = group["effect_high_minus_low"].dropna().to_numpy(float)
        if len(values) == 0:
            continue

        bootstrap = rng.choice(
            values,
            size=(BOOTSTRAP_REPEATS_DIFFICULTY, len(values)),
            replace=True,
        ).mean(axis=1)
        signs = rng.choice(
            np.array([-1.0, 1.0]),
            size=(PERMUTATION_REPEATS_DIFFICULTY, len(values)),
            replace=True,
        )
        null = (signs * values[None, :]).mean(axis=1)
        observed = float(values.mean())
        p_two = float(
            (np.sum(np.abs(null) >= abs(observed)) + 1)
            / (len(null) + 1)
        )

        row = {
            column: key for column, key in zip(group_columns, keys)
        }
        row.update(
            {
                "n_pairs": int(len(values)),
                "mean_effect_high_minus_low": observed,
                "median_effect_high_minus_low": float(np.median(values)),
                "bootstrap_ci_low": float(np.quantile(bootstrap, 0.025)),
                "bootstrap_ci_high": float(np.quantile(bootstrap, 0.975)),
                "sign_flip_p_two_sided": p_two,
                "fraction_positive": float(np.mean(values > 0)),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


def analyze_difficulty_results(
    neural_width: pd.DataFrame,
    count_summary: pd.DataFrame,
    count_diagnostics: pd.DataFrame,
    pairs: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> None:
    neural_effects = pair_effect_rows(
        neural_width,
        pairs,
        value_columns=("A_width_shape", "best_shape_rmse", "mean_excess_ce_H"),
        grouping_columns=("architecture",),
        source="neural",
    )
    atomic_write_csv(paths["tables"] / "neural_pair_effects.csv", neural_effects)

    neural_summary = bootstrap_signflip_summary(
        neural_effects,
        group_columns=("contrast", "source", "architecture", "outcome"),
        cfg=cfg,
    )
    atomic_write_csv(paths["tables"] / "neural_pair_effect_summary.csv", neural_summary)

    count_effects = pair_effect_rows(
        count_summary,
        pairs,
        value_columns=("shape_rmse", "excess_ce_H"),
        grouping_columns=("n_train", "method", "alpha", "beta"),
        source="count_predictor",
    )
    atomic_write_csv(paths["tables"] / "count_pair_effects.csv", count_effects)

    count_summary_effect = bootstrap_signflip_summary(
        count_effects,
        group_columns=(
            "contrast",
            "source",
            "n_train",
            "method",
            "alpha",
            "beta",
            "outcome",
        ),
        cfg=cfg,
    )
    atomic_write_csv(
        paths["tables"] / "count_pair_effect_summary.csv",
        count_summary_effect,
    )

    # K-specific descriptive effects.
    neural_by_k = bootstrap_signflip_summary(
        neural_effects,
        group_columns=("contrast", "architecture", "outcome", "K"),
        cfg=cfg,
    )
    atomic_write_csv(paths["tables"] / "neural_pair_effect_by_K.csv", neural_by_k)

    # Direct association between count sparsity diagnostics and count error.
    h4_diag = count_diagnostics[
        count_diagnostics["h"].astype(int) == cfg.MAX_CONTEXT
    ].copy()
    count_join = count_summary.merge(
        h4_diag[
            [
                "hmm_id",
                "n_train",
                "method",
                "alpha",
                "beta",
                "unseen_test_context_rate",
                "mean_train_count_for_test_context",
                "empirical_effective_context_count",
                "mean_backoff_weight",
            ]
        ],
        on=["hmm_id", "n_train", "method", "alpha", "beta"],
        how="left",
    )
    atomic_write_csv(
        paths["tables"] / "count_error_with_sparsity_diagnostics.csv",
        count_join,
    )

    correlation_rows: List[Dict[str, Any]] = []
    for keys, group in count_join.groupby(
        ["n_train", "method", "alpha", "beta"], dropna=False
    ):
        n_train, method, alpha, beta = keys
        for outcome in ("shape_rmse", "excess_ce_H"):
            for diagnostic in (
                "unseen_test_context_rate",
                "mean_train_count_for_test_context",
                "empirical_effective_context_count",
                "mean_backoff_weight",
            ):
                clean = group[[outcome, diagnostic]].dropna()
                if len(clean) < 4:
                    continue
                rho = clean[outcome].rank().corr(clean[diagnostic].rank())
                correlation_rows.append(
                    {
                        "n_train": int(n_train),
                        "method": method,
                        "alpha": float(alpha),
                        "beta": beta,
                        "outcome": outcome,
                        "diagnostic": diagnostic,
                        "spearman_rho": float(rho),
                        "n_hmms": int(len(clean)),
                    }
                )
    atomic_write_csv(
        paths["tables"] / "count_sparsity_correlations.csv",
        pd.DataFrame(correlation_rows),
    )

    create_difficulty_figures(
        neural_width,
        count_summary,
        count_diagnostics,
        pairs,
        cfg,
        paths,
    )


def create_difficulty_figures(
    neural_width: pd.DataFrame,
    count_summary: pd.DataFrame,
    count_diagnostics: pd.DataFrame,
    pairs: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> None:
    # Figure 1: Count-predictor excess CE versus sample size for occupancy pairs.
    primary_count = count_summary[
        (count_summary["alpha"].astype(float) == PRIMARY_NGRAM_ALPHA)
        & (
            (count_summary["method"] == "additive")
            | (
                (count_summary["method"] == "interpolated_backoff")
                & (count_summary["beta"].astype(float) == PRIMARY_BACKOFF_BETA)
            )
        )
    ].copy()
    occupancy_ids = set(
        pairs[pairs["contrast"] == "occupancy_contrast"][
            ["hmm_low", "hmm_high"]
        ].astype(str).to_numpy().reshape(-1)
    )
    plot_data = primary_count[
        primary_count["hmm_id"].astype(str).isin(occupancy_ids)
    ]
    if not plot_data.empty:
        grouped = (
            plot_data.groupby(["method", "pair_role", "n_train"], as_index=False)
            .agg(mean_excess=("excess_ce_H", "mean"), se=("excess_ce_H", "sem"))
        )
        plt.figure(figsize=(8, 5))
        for (method, role), group in grouped.groupby(["method", "pair_role"]):
            group = group.sort_values("n_train")
            plt.errorbar(
                group["n_train"],
                group["mean_excess"],
                yerr=group["se"],
                marker="o",
                capsize=3,
                label=f"{method}: {role} occupancy",
            )
        plt.xscale("log", base=2)
        plt.xlabel("Number of training sequences")
        plt.ylabel("Excess CE at h=4")
        plt.title("Count-predictor difficulty under context-occupancy contrast")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            paths["figures"] / "count_occupancy_excess_vs_sample_size.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close()

    # Figure 2: Neural A_width pair effects.
    neural_effects = pair_effect_rows(
        neural_width,
        pairs,
        value_columns=("A_width_shape",),
        grouping_columns=("architecture",),
        source="neural",
    )
    if not neural_effects.empty:
        plt.figure(figsize=(8, 5))
        positions = []
        labels = []
        data = []
        position = 1
        for contrast in ("occupancy_contrast", "gain_contrast"):
            for architecture in cfg.ARCHITECTURES:
                values = neural_effects[
                    (neural_effects["contrast"] == contrast)
                    & (neural_effects["architecture"] == architecture)
                ]["effect_high_minus_low"].dropna().to_numpy(float)
                if len(values):
                    positions.append(position)
                    labels.append(
                        f"{contrast.replace('_contrast', '')}\n{architecture}"
                    )
                    data.append(values)
                    position += 1
        if data:
            plt.boxplot(data, positions=positions, labels=labels, showmeans=True)
            plt.axhline(0.0, linewidth=1)
            plt.ylabel("High minus low A_width_shape")
            plt.title("Matched neural recovery effects")
            plt.tight_layout()
            plt.savefig(
                paths["figures"] / "neural_matched_pair_effects.png",
                dpi=220,
                bbox_inches="tight",
            )
        plt.close()

    # Figure 3: Unseen-context rate versus count excess CE.
    h4 = count_diagnostics[
        (count_diagnostics["h"].astype(int) == cfg.MAX_CONTEXT)
        & (count_diagnostics["alpha"].astype(float) == PRIMARY_NGRAM_ALPHA)
        & (count_diagnostics["method"] == "additive")
        & (count_diagnostics["n_train"].astype(int) == NEURAL_N_TRAIN)
    ]
    count_primary = count_summary[
        (count_summary["alpha"].astype(float) == PRIMARY_NGRAM_ALPHA)
        & (count_summary["method"] == "additive")
        & (count_summary["n_train"].astype(int) == NEURAL_N_TRAIN)
    ]
    scatter = count_primary.merge(
        h4[["hmm_id", "unseen_test_context_rate"]],
        on="hmm_id",
        how="left",
    )
    if not scatter.empty:
        plt.figure(figsize=(7, 5))
        plt.scatter(
            scatter["unseen_test_context_rate"],
            scatter["excess_ce_H"],
            alpha=0.8,
        )
        plt.xlabel("Unseen h=4 test-context rate")
        plt.ylabel("Additive n-gram excess CE")
        plt.title("Context sparsity and count-predictor error")
        plt.tight_layout()
        plt.savefig(
            paths["figures"] / "unseen_context_rate_vs_count_error.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close()


# -----------------------------------------------------------------------------
# Protocol and completion records
# -----------------------------------------------------------------------------

def write_difficulty_protocol(cfg: Config, paths: Dict[str, Path]) -> None:
    protocol = {
        "study": "predictor_specific_difficulty",
        "version": cfg.MATCH_PROTOCOL_VERSION,
        "source_confirmatory_root": str(SOURCE_CONFIRM_ROOT),
        "selection": {
            "outcome_blind": True,
            "exclude_previous_confirmatory_hmms": True,
            "K_values": list(cfg.K_VALUES),
            "pairs_per_K_per_contrast": (
                1 if DIFFICULTY_SMOKE_TEST else PAIRS_PER_K_PER_CONTRAST
            ),
            "contrasts": {
                "occupancy_contrast": {
                    "description": (
                        "Effective length-4 context count differs while gain, rank, "
                        "sigma, mixing, Bayes CE, spectral signal, and min-symbol "
                        "probability are controlled. Symbol entropy is recorded but "
                        "not matched because it is structurally coupled to H(U_4) in "
                        "this generator."
                    ),
                    "tiers": list(OCCUPANCY_MATCH_TIERS),
                },
                "gain_contrast": {
                    "description": (
                        "Context gain differs while length-4 context occupancy, "
                        "symbol entropy, min-symbol probability, rank, sigma, mixing, "
                        "Bayes CE, and spectral signal are controlled."
                    ),
                    "tiers": list(GAIN_MATCH_TIERS),
                },
            },
        },
        "neural_grid": {
            "architectures": list(cfg.ARCHITECTURES),
            "dimensions": list(cfg.DIMENSIONS),
            "seeds": list(cfg.SEEDS),
            "n_train": cfg.N_TRAIN_MAIN,
            "expected_runs": (
                2
                * (2 * len(cfg.K_VALUES) * (1 if DIFFICULTY_SMOKE_TEST else PAIRS_PER_K_PER_CONTRAST))
                * len(cfg.ARCHITECTURES)
                * len(cfg.DIMENSIONS)
                * len(cfg.SEEDS)
            ),
        },
        "count_predictors": {
            "n_train_values": list(NGRAM_TRAIN_SIZES),
            "alphas": list(NGRAM_ALPHAS),
            "methods": ["additive", "interpolated_backoff"],
            "backoff_betas": list(NGRAM_BACKOFF_BETAS),
            "primary_alpha": PRIMARY_NGRAM_ALPHA,
            "primary_backoff_beta": PRIMARY_BACKOFF_BETA,
        },
        "primary_hypotheses": {
            "H1": (
                "Higher context occupancy increases count-predictor error under "
                "matched predictive-profile controls."
            ),
            "H2": (
                "Higher predictive gain increases neural Bayes-shape recovery error "
                "when context occupancy and marginal entropy are matched."
            ),
            "H3": (
                "The occupancy effect on count predictors weakens with sample size "
                "and interpolated backoff."
            ),
        },
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    atomic_write_json(paths["config"] / "DIFFICULTY_PROTOCOL.json", protocol)


# -----------------------------------------------------------------------------
# Main entry point
# -----------------------------------------------------------------------------

def predictor_specific_difficulty_main() -> None:
    cfg = make_difficulty_config()
    mount_drive_if_needed(cfg)
    configure_local_paths_for_non_colab(cfg)

    # configure_local_paths_for_non_colab changes only cfg.PROJECT_ROOT. The source
    # candidate bank is a real Drive dependency for the full run.
    paths = project_paths(cfg)
    logger = setup_logger(paths["logs"])
    device = resolve_device(cfg)
    set_global_seed(cfg.MASTER_SEED, cfg.DETERMINISTIC_TORCH)
    save_run_manifest(cfg, paths, device)
    write_difficulty_protocol(cfg, paths)

    pairs_per_cell = 1 if DIFFICULTY_SMOKE_TEST else PAIRS_PER_K_PER_CONTRAST
    expected_pairs = 2 * len(cfg.K_VALUES) * pairs_per_cell
    expected_hmms = 2 * expected_pairs
    expected_neural = (
        expected_hmms
        * len(cfg.ARCHITECTURES)
        * len(cfg.DIMENSIONS)
        * len(cfg.SEEDS)
    )

    logger.info("Predictor-specific difficulty root: %s", paths["root"])
    logger.info("Device: %s", device)
    logger.info(
        "Requested design: %d pairs, %d HMMs, %d neural conditions",
        expected_pairs,
        expected_hmms,
        expected_neural,
    )

    bank = ensure_bank_marginals_and_context(cfg, paths, logger)
    selected_design, pairs = select_difficulty_pairs(
        bank, cfg, paths, logger
    )

    selected_full = materialize_selected_hmms(
        selected_design, cfg, paths, logger
    )
    selected_full = add_selected_context_and_marginals(
        selected_full, selected_design, cfg
    )
    atomic_write_csv(
        paths["selected"] / "difficulty_selected_hmm_metrics_full.csv",
        selected_full,
    )

    if len(selected_full) != expected_hmms:
        raise RuntimeError(
            f"Expected {expected_hmms} selected HMMs, found {len(selected_full)}."
        )

    dataset_paths = prepare_all_datasets(
        selected_full, cfg, paths, logger
    )
    atomic_write_json(
        paths["root"] / "DIFFICULTY_PREPARATION_COMPLETE.json",
        {
            "status": "difficulty_preparation_completed",
            "pairs": int(len(pairs)),
            "selected_hmms": int(len(selected_full)),
            "expected_neural_conditions": int(expected_neural),
            "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
    )

    if PREPARE_ONLY_DIFFICULTY:
        logger.info("PREPARE_ONLY_DIFFICULTY=True; stopping before learner runs")
        return

    count_summary = pd.DataFrame()
    count_diagnostics = pd.DataFrame()
    if RUN_NGRAM_DIFFICULTY:
        count_summary, count_diagnostics = run_count_predictor_experiment(
            selected_full,
            dataset_paths,
            cfg,
            paths,
            logger,
        )

    if RUN_NEURAL_DIFFICULTY:
        completed, expected = run_difficulty_neural_grid(
            selected_full,
            dataset_paths,
            cfg,
            paths,
            logger,
            device,
        )
        if completed < expected:
            atomic_write_json(
                paths["root"] / "DIFFICULTY_STATUS.json",
                {
                    "status": "difficulty_neural_training_incomplete",
                    "done_runs": int(completed),
                    "expected_runs": int(expected),
                    "remaining_runs": int(expected - completed),
                    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                },
            )
            logger.info(
                "Neural training is incomplete. Rerun the same script to resume."
            )
            return

    raw = collect_done_results(paths["runs"])
    atomic_write_csv(
        paths["tables"] / "difficulty_neural_training_raw.csv",
        raw,
    )
    run_metrics = attach_recovery_metrics(
        raw, selected_full, cfg
    )
    atomic_write_csv(
        paths["tables"] / "difficulty_neural_training_with_metrics.csv",
        run_metrics,
    )
    neural_width = width_summary(run_metrics, cfg).merge(
        selected_full[
            ["hmm_id", "K", "pair_id", "contrast", "pair_role"]
        ],
        on="hmm_id",
        how="left",
    )
    atomic_write_csv(
        paths["tables"] / "difficulty_neural_width_summary.csv",
        neural_width,
    )

    if (
        RUN_ANALYSIS_DIFFICULTY
        and not neural_width.empty
        and not count_summary.empty
        and not count_diagnostics.empty
    ):
        analyze_difficulty_results(
            neural_width,
            count_summary,
            count_diagnostics,
            pairs,
            cfg,
            paths,
        )

    completion = {
        "status": "predictor_specific_difficulty_completed",
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "root": str(paths["root"]),
        "pairs": int(len(pairs)),
        "selected_hmms": int(len(selected_full)),
        "expected_neural_runs": int(expected_neural),
        "done_neural_runs": int(len(raw)),
        "count_summary_rows": int(len(count_summary)),
        "count_diagnostic_rows": int(len(count_diagnostics)),
    }
    atomic_write_json(
        paths["root"] / "DIFFICULTY_PIPELINE_COMPLETE.json",
        completion,
    )
    logger.info("Predictor-specific difficulty experiment finished: %s", completion)


if __name__ == "__main__":
    predictor_specific_difficulty_main()